### Import images and labels

In [ ]:
import csv
import pickle
import numpy as np
from skimage import io
from scipy.ndimage import binary_fill_holes

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
TARGET_SCALE = "min_234"
MIN_CELL_AREA = 10000

DATASET_CONFIGS = {
    1: {
        "image_path": "Lysosome_masked_crops_1024.tif",
        "csv_path": "PhTau_histogram_clusters.csv",
        "coord_path": "processed_data/spot_coordinates.pkl",
        "delaunay_path": "processed_data/delaunay_results.pkl",
    },
    2: {
        "image_path": "cell_gallery_1024x1024_6ch.tif",
        "csv_path": "gallery_phtau_ground_truth.csv",
        "coord_path": "processed_data_2/spot_coordinates.pkl",
        "delaunay_path": "processed_data_2/delaunay_results.pkl",
    },
}

# Master data containers
images = {}
n_images = {}
labels = {}
categories = {}
coord_map = {}
delaunay_objects_dict = {}

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def load_csv_labels(csv_path, total_expected_frames):
    """Parses cluster labels from CSV and pads remaining frames as untreated."""
    parsed_labels = []
    with open(csv_path, "r") as csvfile:
        reader = csv.reader(csvfile)
        next(reader)  # Skip header row
        for row in reader:
            val = row[-1]
            if val == "0":
                parsed_labels.append(["0", "dox+\nPhTau+"])
            elif val == "1":
                parsed_labels.append(["1", "dox+\nPhTau-"])

    # Pad missing indices up to the total stack size (assumed untreated)
    while len(parsed_labels) < total_expected_frames:
        parsed_labels.append(["2", "dox-"])
        
    return parsed_labels


def load_pickle_scale(pickle_path, scale_key):
    """Helper to extract a specific resolution scale from a pickle file."""
    with open(pickle_path, "rb") as f:
        return pickle.load(f)[scale_key]


def compute_valid_mask_indices(image_stack, area_threshold):
    """ Vectorised masking and area filter over a standardised image stack."""
    # Since channel is guaranteed to be at axis -1, collapse it if 4D
    raw_masks = np.any(image_stack > 0, axis=-1) if image_stack.ndim == 4 else image_stack > 0

    # Fill internal segmentation holes slice-by-slice across the stack
    cell_masks = binary_fill_holes(raw_masks)
    
    # Calculate pixel surface areas by summing spatial dimensions (axes 1 and 2)
    cell_areas = np.sum(cell_masks, axis=(1, 2))
    
    # Return boolean array masking frames that meet size criteria
    return cell_areas >= area_threshold

# ==========================================
# 3. PIPELINE EXECUTION
# ==========================================
for idx, config in DATASET_CONFIGS.items():
    # Load image stack pipeline
    raw_images_stack = io.imread(config["image_path"])

    # Standardise 4D stacks to always be (frames, spatial_1, spatial_2, channels)
    if raw_images_stack.ndim == 4:
        # If your channel is currently at axis 1 (e.g., shape 100, 6, 1024, 1024)
        channel_axis = np.argmin(raw_images_stack.shape)
        if channel_axis != -1:
            raw_images_stack = np.moveaxis(raw_images_stack, channel_axis, -1)
    
    total_frames = raw_images_stack.shape[0]

    # Load and map structural metadata sequentially
    all_labels = load_csv_labels(config["csv_path"], total_frames)
    full_coord = load_pickle_scale(config["coord_path"], TARGET_SCALE)
    full_delaunay = load_pickle_scale(config["delaunay_path"], TARGET_SCALE)

    # Compute valid mask filter indices using vectorised area sums
    valid_indices = compute_valid_mask_indices(raw_images_stack, MIN_CELL_AREA)

    # Identify the rejected indices where the condition is False
    rejected_indices = np.where(~valid_indices)[0]
    
    # Print the rejected cell indices for the current dataset
    print(f"Dataset {idx} ({config['image_path']}):")
    if len(rejected_indices) > 0:
        print(f"  Rejected {len(rejected_indices)} cells due to small area (< {MIN_CELL_AREA} px).")
        print(f"  Rejected Indices: {rejected_indices.tolist()}\n")
    else:
        print("  No cells were rejected.\n")

    # Apply boolean index filtering to numerical arrays
    images[idx] = raw_images_stack[valid_indices]
    n_images[idx] = np.sum(valid_indices)
    
    # Filter Python list metadata using matching list comprehensions
    labels[idx] = [all_labels[i] for i, valid in enumerate(valid_indices) if valid]
    coord_map[idx] = [full_coord[i] for i, valid in enumerate(valid_indices) if valid]
    delaunay_objects_dict[idx] = [full_delaunay[i] for i, valid in enumerate(valid_indices) if valid]

    # Parse remaining categories, sorting by category id descending
    unique_pairs = {tuple(item) for item in labels[idx]}
    sorted_pairs = sorted(unique_pairs, key=lambda x: int(x[0]), reverse=True)
    categories[idx] = [pair[1] for pair in sorted_pairs]

## Analysis by dox/PhTau status

### Cell areas

In [ ]:
import numpy as np
from scipy.ndimage import binary_fill_holes

def extract_cell_areas_vectorised(image_stack):
    """
    Calculate surface areas for a stack of cell images.
    Supports 3D (frames, H, W) and 4D (frames, H, W, channels) arrays.
    """
    if image_stack.ndim == 0 or image_stack.shape[0] == 0:
        return []

    # Binarise and collapse channels if 4D
    if image_stack.ndim == 4:
        raw_masks = np.any(image_stack > 0, axis=-1)
    else:
        raw_masks = image_stack > 0

    # Fill internal holes within the cell masks
    cell_masks = binary_fill_holes(raw_masks)
    
    # Calculate area by summing active pixels across spatial dimensions
    cell_areas = np.sum(cell_masks, axis=(1, 2)).astype(float)
    
    return cell_areas.tolist()


# Extract areas from pre-filtered image sets
areas_1 = extract_cell_areas_vectorised(images[1])
areas_2 = extract_cell_areas_vectorised(images[2])

print(f"Dataset 1: Extracted {len(areas_1)} cell areas.")
print(f"Dataset 2: Extracted {len(areas_2)} cell areas.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Traffic light colour mapping
traffic_light_map = {
    "dox-": "#2ecc71",          # Green
    "dox+\nPhTau-": "#f1c40f",  # Yellow
    "dox+\nPhTau+": "#e74c3c"   # Red
} 

# Define pixel width and calculate the area conversion factor (µm² per pixel)
pixel_width = 0.0739859
area_conversion = pixel_width ** 2

# Extract index 1 to get the string label and convert area to µm²
df1 = pd.DataFrame({
    'Area_um2': [a * area_conversion for a in areas_1],
    'Condition': [label[1] for label in labels[1]]
}).dropna()

df2 = pd.DataFrame({
    'Area_um2': [a * area_conversion for a in areas_2],
    'Condition': [label[1] for label in labels[2]]
}).dropna()

# Combine both datasets
df_combined = pd.concat([df1, df2], ignore_index=True)

# Condition orders
order_1 = ['dox-', 'dox+\nPhTau-', 'dox+\nPhTau+']
order_2 = ['dox+\nPhTau-', 'dox+\nPhTau+']
full_order = ['dox-', 'dox+\nPhTau-', 'dox+\nPhTau+']

# --- Plot 1: Dataset 1 and Dataset 2 Side-by-Side ---
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

datasets = [
    (df1, order_1, 'Cell Areas (Dataset 1)', axes[0]), 
    (df2, order_2, 'Cell Areas (Dataset 2)', axes[1])
]

for df_subset, order_subset, title_text, ax in datasets:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x='Condition',
        y='Area_um2',
        hue='Condition',
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )
    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel("Area (µm²)", fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig("Figures/areas_by_dataset.svg", format="svg", bbox_inches="tight")
plt.show()

# --- Plot 2: Separate Combined Plot ---
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined,
    x='Condition',
    y='Area_um2',
    hue='Condition',
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1
)
ax.set_title("Cell Areas", fontsize=14, pad=10)
ax.set_ylabel("Area (µm²)", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.savefig("Figures/areas_combined.svg", format="svg", bbox_inches="tight")
plt.show()

# Summary Statistics (rounded to 2 decimal places for better precision with physical units)
print("--- Dataset 1 Summary Statistics (µm²) ---")
print(df1.groupby('Condition')['Area_um2'].describe().round(2))

print("\n--- Dataset 2 Summary Statistics (µm²) ---")
print(df2.groupby('Condition')['Area_um2'].describe().round(2))

print("\n--- Combined Dataset Summary Statistics (µm²) ---")
print(df_combined.groupby('Condition')['Area_um2'].describe().round(2))

In [ ]:
from itertools import combinations

from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests


def run_group_tests(
    df,
    value_col,
    dataset_name,
    metric_name,
    condition_order,
    group_col="Condition",
    alpha=0.05
):
    """
    Run a Kruskal–Wallis test followed by pairwise Mann–Whitney U tests.

    Parameters
    ----------
    df : pandas.DataFrame
        Input DataFrame.
    value_col : str
        Numeric measurement column.
    dataset_name : str
        Dataset name printed in the output.
    metric_name : str
        Human-readable measurement name.
    condition_order : list
        Desired order of groups.
    group_col : str, default="Condition"
        Column containing the group labels.
    alpha : float, default=0.05
        Significance threshold.
    """

    # Validate required columns
    missing_columns = [
        column for column in [value_col, group_col]
        if column not in df.columns
    ]

    if missing_columns:
        raise KeyError(
            f"Missing column(s): {missing_columns}. "
            f"Available columns are: {list(df.columns)}"
        )

    print(f"\n{'=' * 75}")
    print(f"{metric_name} — {dataset_name}")
    print(f"{'=' * 75}")

    # Extract non-empty groups in the requested order
    groups = {
        group: df.loc[
            df[group_col] == group,
            value_col
        ].dropna()
        for group in condition_order
    }

    # Remove groups with no observations
    groups = {
        group: values
        for group, values in groups.items()
        if len(values) > 0
    }

    print("\nSample sizes:")
    for group, values in groups.items():
        print(f"  {group}: n = {len(values)}")

    if len(groups) < 2:
        print("\nNot enough groups for statistical testing.")
        return None

    # Kruskal–Wallis omnibus test
    kw_stat, kw_p = kruskal(*groups.values())

    print("\nKruskal–Wallis test:")
    print(f"  H = {kw_stat:.4f}")
    print(f"  p = {kw_p:.4g}")

    if kw_p < alpha:
        print(f"  Significant overall difference at alpha = {alpha}.")
    else:
        print(f"  No significant overall difference at alpha = {alpha}.")

    # Pairwise Mann–Whitney U tests
    pairwise_results = []

    for group_a, group_b in combinations(groups.keys(), 2):
        values_a = groups[group_a]
        values_b = groups[group_b]

        u_stat, raw_p = mannwhitneyu(
            values_a,
            values_b,
            alternative="two-sided",
            method="asymptotic"
        )

        pairwise_results.append({
            "Metric": metric_name,
            "Dataset": dataset_name,
            "Group column": group_col,
            "Comparison": f"{group_a} vs {group_b}",
            "U": u_stat,
            "Raw_p": raw_p
        })

    # Holm correction for multiple comparisons
    raw_p_values = [
        result["Raw_p"]
        for result in pairwise_results
    ]

    _, adjusted_p_values, _, _ = multipletests(
        raw_p_values,
        method="holm",
        alpha=alpha
    )

    for result, adjusted_p in zip(
        pairwise_results,
        adjusted_p_values
    ):
        result["Holm_adjusted_p"] = adjusted_p
        result["Significant"] = adjusted_p < alpha

    results_df = pd.DataFrame(pairwise_results)

    print("\nPairwise Mann–Whitney U tests with Holm correction:")
    print(
        results_df.to_string(
            index=False,
            formatters={
                "U": "{:.2f}".format,
                "Raw_p": "{:.4g}".format,
                "Holm_adjusted_p": "{:.4g}".format
            }
        )
    )

    return results_df

results_area1 = run_group_tests(
    df=df1,
    value_col="Area_um2",
    dataset_name="Dataset 1",
    metric_name="Cell area",
    condition_order=order_1
)

results_area2 = run_group_tests(
    df=df2,
    value_col="Area_um2",
    dataset_name="Dataset 2",
    metric_name="Cell area",
    condition_order=order_2
)

results_area_combined = run_group_tests(
    df=df_combined,
    value_col="Area_um2",
    dataset_name="Combined dataset",
    metric_name="Cell area",
    condition_order=full_order
)

### Lysosome counts

In [ ]:
import numpy as np


def extract_lysosome_counts(coord_map_dict):
    """Calculate the number of lysosomes (spots) for each image frame.

    Supports coordinate maps structured as lists of arrays/lists.
    """
    # Return empty list if the dataset key does not exist or is empty
    if coord_map_dict is None or len(coord_map_dict) == 0:
        return []

    # Count the number of coordinate points (spots) in each image frame
    lysosome_counts = np.array([len(img_spots) for img_spots in coord_map_dict])

    return lysosome_counts.tolist()


# Extract lysosome spot counts from pre-filtered coordinate maps
lysosomes_1 = extract_lysosome_counts(coord_map.get(1, []))
lysosomes_2 = extract_lysosome_counts(coord_map.get(2, []))

print(f"Dataset 1: Extracted {len(lysosomes_1)} frame spot counts.")
print(f"Dataset 2: Extracted {len(lysosomes_2)} frame spot counts.")

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Build DataFrames using lysosome counts and condition labels
df1_lys = pd.DataFrame(
    {
        "Lysosome_Count": lysosomes_1,
        "Condition": [label[1] for label in labels[1]],
    }
).dropna()

df2_lys = pd.DataFrame(
    {
        "Lysosome_Count": lysosomes_2,
        "Condition": [label[1] for label in labels[2]],
    }
).dropna()

# Combine both datasets
df_combined_lys = pd.concat([df1_lys, df2_lys], ignore_index=True)

# Define orders to maintain color alignment across all plots
order_1 = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
order_2 = ["dox+\nPhTau-", "dox+\nPhTau+"]
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

# --- Plot 1: Dataset 1 and Dataset 2 Side-by-Side ---
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

datasets = [
    (df1_lys, order_1, 'Lysosome Counts (Dataset 1)', axes[0]), 
    (df2_lys, order_2, 'Lysosome Counts (Dataset 2)', axes[1])
]

for df_subset, order_subset, title_text, ax in datasets:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y="Lysosome_Count",
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )
    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel("Lysosomes per Cell", fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig("Figures/lysosome_counts_by_dataset.svg", format="svg", bbox_inches="tight")
plt.show()

# --- Plot 2: Separate Combined Plot ---
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined_lys,
    x="Condition",
    y="Lysosome_Count",
    hue="Condition",
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1
)
ax.set_title("Lysosome Counts", fontsize=14, pad=10)
ax.set_ylabel("Lysosomes per Cell", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.savefig("Figures/lysosome_counts_combined.svg", format="svg", bbox_inches="tight")
plt.show()

# 4. Display summary statistics
print("--- Dataset 1 Lysosome Summary Statistics ---")
print(df1_lys.groupby("Condition")["Lysosome_Count"].describe().round(1))

print("\n--- Dataset 2 Lysosome Summary Statistics ---")
print(df2_lys.groupby("Condition")["Lysosome_Count"].describe().round(1))

print("\n--- Combined Dataset Lysosome Summary Statistics ---")
print(df_combined_lys.groupby("Condition")["Lysosome_Count"].describe().round(1))

In [ ]:
results_lys1 = run_group_tests(
    df=df1_lys,
    value_col="Lysosome_Count",
    dataset_name="Dataset 1",
    metric_name="Lysosome count",
    condition_order=order_1
)

results_lys2 = run_group_tests(
    df=df2_lys,
    value_col="Lysosome_Count",
    dataset_name="Dataset 2",
    metric_name="Lysosome count",
    condition_order=order_2
)

results_lys_combined = run_group_tests(
    df=df_combined_lys,
    value_col="Lysosome_Count",
    dataset_name="Combined dataset",
    metric_name="Lysosome count",
    condition_order=full_order
)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# --- CONVERSION FACTOR & CATEGORICAL SETUP ---
pixel_width = 0.0739859  # µm
pixel_area_um2 = pixel_width**2  # Area of one pixel in µm²

class_ids = [2, 1, 0]  # Order mapping for numerical labels
class_names = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
label_to_condition = dict(zip(class_ids, class_names))

# --- DATA EXTRACTION & DENSITY CALCULATION ---
# Dataset 1
counts_d1 = np.array([len(img_spots) for img_spots in coord_map[1]])
areas_d1_um2 = (
    np.array(extract_cell_areas_vectorised(images[1])) * pixel_area_um2
)
density_d1 = np.divide(
    counts_d1,
    areas_d1_um2,
    out=np.zeros_like(counts_d1, dtype=float),
    where=areas_d1_um2 > 0,
)
labels_d1 = np.array([int(l[0]) for l in labels[1]])

# Dataset 2
counts_d2 = np.array([len(img_spots) for img_spots in coord_map[2]])
areas_d2_um2 = (
    np.array(extract_cell_areas_vectorised(images[2])) * pixel_area_um2
)
density_d2 = np.divide(
    counts_d2,
    areas_d2_um2,
    out=np.zeros_like(counts_d2, dtype=float),
    where=areas_d2_um2 > 0,
)
labels_d2 = np.array([int(l[0]) for l in labels[2]])

# --- 1. BUILD DATAFRAMES MATCHING TEMPLATE ---
df1_density = pd.DataFrame(
    {
        "Lysosome_Density": density_d1,
        "Condition": [label_to_condition[lbl] for lbl in labels_d1],
    }
).dropna()

df2_density = pd.DataFrame(
    {
        "Lysosome_Density": density_d2,
        "Condition": [label_to_condition[lbl] for lbl in labels_d2],
    }
).dropna()

# Combine both datasets
df_combined_density = pd.concat([df1_density, df2_density], ignore_index=True)

# Define orders to maintain colour alignment across all plots
order_1 = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
order_2 = ["dox+\nPhTau-", "dox+\nPhTau+"]
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

# --- 2. SET UP PLOTTING CANVAS (Side-by-side plot for Dataset 1 and Dataset 2) ---
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

datasets = [
    (df1_density, order_1, 'Lysosome Density (Dataset 1)', axes[0]), 
    (df2_density, order_2, 'Lysosome Density (Dataset 2)', axes[1])
]

for df_subset, order_subset, title_text, ax in datasets:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y="Lysosome_Density",
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1
    )
    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel("Lysosomes per µm²", fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig("Figures/lysosome_density_by_dataset.svg", format="svg", bbox_inches="tight")
plt.show()

# --- 3. GENERATE AND SAVE THE COMBINED PLOT INDEPENDENTLY ---
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined_density,
    x="Condition",
    y="Lysosome_Density",
    hue="Condition",
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1
)
ax.set_title("Lysosome Density", fontsize=14, pad=10)
ax.set_ylabel("Lysosomes per µm²", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.savefig("Figures/lysosome_density_combined.svg", format="svg", bbox_inches="tight")
plt.show()

# --- 4. DISPLAY SUMMARY STATISTICS ---
print("--- Dataset 1 Lysosome Density Summary Statistics ---")
print(df1_density.groupby("Condition")["Lysosome_Density"].describe().round(4))

print("\n--- Dataset 2 Lysosome Density Summary Statistics ---")
print(df2_density.groupby("Condition")["Lysosome_Density"].describe().round(4))

print("\n--- Combined Dataset Lysosome Density Summary Statistics ---")
print(
    df_combined_density.groupby("Condition")["Lysosome_Density"]
    .describe()
    .round(4)
)

In [ ]:
# Statistical testing for lysosome density

results_density1 = run_group_tests(
    df=df1_density,
    value_col="Lysosome_Density",
    dataset_name="Dataset 1",
    metric_name="Lysosome density",
    condition_order=order_1
)

results_density2 = run_group_tests(
    df=df2_density,
    value_col="Lysosome_Density",
    dataset_name="Dataset 2",
    metric_name="Lysosome density",
    condition_order=order_2
)

results_density_combined = run_group_tests(
    df=df_combined_density,
    value_col="Lysosome_Density",
    dataset_name="Combined dataset",
    metric_name="Lysosome density",
    condition_order=full_order
)

### Lysosome coverage areas

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_mean
from skimage.filters import threshold_li      # Fiji "Default" approximation
from skimage.filters import threshold_otsu    # Added for robust bimodal/long-tail
from skimage.filters import threshold_triangle# Added specifically for single-peak long tails


def calculate_lysosome_coverage(image_stack, lysosome_channel):
    """
    Calculate lysosome coverage per cell using robust thresholding algorithms
    (Li, Otsu, Triangle, Mean) designed for punctate, long-tailed distributions.
    """
    if image_stack.ndim == 3:
        default_channels = 6 
        n_cells = image_stack.shape[0] // default_channels
        image_stack = image_stack.reshape(n_cells, default_channels, image_stack.shape[1], image_stack.shape[2])

    results = []

    for cell_id, img in enumerate(image_stack):
        channel_axis = np.argmin(img.shape)
        
        # 1. Generate the initial raw mask
        raw_mask = np.any(img > 0, axis=channel_axis)
        
        # 2. Fill internal holes
        cell_mask = binary_fill_holes(raw_mask)
        
        # 3. Calculate true area of the solid cell body
        cell_area = np.sum(cell_mask)

        # 4. Extract lysosome image safely based on axis layout
        if channel_axis == 0:
            lys_img = img[lysosome_channel].astype(float)
        else:
            lys_img = img[:, :, lysosome_channel].astype(float)

        # Only consider pixels inside cell
        lys_pixels = lys_img[cell_mask]

        if lys_pixels.size == 0:
            continue

        # 5. Calculate robust thresholds
        try:
            thresholds = {
                "li": threshold_li(lys_pixels),
                "otsu": threshold_otsu(lys_pixels),
                "triangle": threshold_triangle(lys_pixels),
                "mean": threshold_mean(lys_pixels)
            }
        except Exception as e:
            print(f"[WARNING] Cell ID {cell_id} failed threshold calculation. Error: {e}")
            thresholds = {"li": np.nan, "otsu": np.nan, "triangle": np.nan, "mean": np.nan}

        row = {
            "cell_id": cell_id,
            "cell_area_px": cell_area
        }

        for method, thr in thresholds.items():
            if np.isnan(thr):
                row[f"{method}_threshold"] = np.nan
                row[f"{method}_lys_area_px"] = np.nan
                row[f"{method}_coverage_pct"] = np.nan
                continue

            lys_mask = (lys_img > thr) & cell_mask
            lys_area = np.sum(lys_mask)
            coverage = 100 * lys_area / cell_area

            row[f"{method}_threshold"] = thr
            row[f"{method}_lys_area_px"] = lys_area
            row[f"{method}_coverage_pct"] = coverage

        results.append(row)

    return pd.DataFrame(results)


# Execute and print updated summary metrics
df1 = calculate_lysosome_coverage(images[1], lysosome_channel=0)
df2 = calculate_lysosome_coverage(images[2], lysosome_channel=0)

target_metrics = ["li_coverage_pct", "otsu_coverage_pct", "triangle_coverage_pct", "mean_coverage_pct"]

print("DF1 Summary:")
print(df1[target_metrics].describe())

print("\nDF2 Summary:")
print(df2[target_metrics].describe())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_li, threshold_otsu, threshold_triangle, threshold_mean

def load_fiji_lut(filepath):
    """Parses a standard 768-byte binary Fiji LUT file into a Matplotlib colormap."""
    with open(filepath, 'rb') as f:
        lut_data = np.frombuffer(f.read(), dtype=np.uint8)
    
    if lut_data.size != 768:
        raise ValueError("Invalid Fiji LUT file size. Expected exactly 768 bytes.")
    
    lut = lut_data.reshape(3, 256).T / 255.0
    return LinearSegmentedColormap.from_list("fiji_custom", lut)

img = images[1][0]
channel_axis = np.argmin(img.shape)

# 1. Generate initial raw mask and fill holes
raw_mask = np.any(img > 0, axis=channel_axis)
cell_mask = binary_fill_holes(raw_mask)
cell_area = np.sum(cell_mask)

# 2. Extract lysosome image safely
if channel_axis == 0:
    lys_img = img[0].astype(float)
else:
    lys_img = img[:, :, 0].astype(float)

lys_pixels = lys_img[cell_mask]

# 3. Calculate thresholds
methods = {
    "li": threshold_li,
    "otsu": threshold_otsu,
    "triangle": threshold_triangle,
    "mean": threshold_mean
}

thresholds = {}
for name, func in methods.items():
    try:
        thresholds[name] = func(lys_pixels) if lys_pixels.size > 0 else np.nan
    except Exception as e:
        print(f"[WARNING] Method '{name}' failed. Error: {e}")
        thresholds[name] = np.nan

row = {"cell_id": 0, "cell_area_px": cell_area}
masks = {}
for method, thr in thresholds.items():
    if np.isnan(thr):
        row[f"{method}_threshold"] = np.nan
        row[f"{method}_lys_area_px"] = np.nan
        row[f"{method}_coverage_pct"] = np.nan
        masks[method] = np.zeros_like(cell_mask, dtype=bool)
        continue

    lys_mask = (lys_img > thr) & cell_mask
    lys_area = np.sum(lys_mask)
    coverage = 100 * lys_area / cell_area if cell_area > 0 else 0

    row[f"{method}_threshold"] = thr
    row[f"{method}_lys_area_px"] = lys_area
    row[f"{method}_coverage_pct"] = coverage
    masks[method] = lys_mask

# 4. Load your Fiji LUT file
fiji_cmap = load_fiji_lut("lysosomes.lut")

# 5. Plot comparison in two rows:
#    Row 1: lysosome image and cell-body mask
#    Row 2: thresholded lysosome masks

valid_methods = [m for m, thr in thresholds.items() if not np.isnan(thr)]

# Four threshold methods are expected; ensure enough columns if methods change
n_cols = max(4, len(valid_methods))

fig, axes = plt.subplots(
    2,
    n_cols,
    figsize=(4.5 * n_cols, 9),
    facecolor="white",
    gridspec_kw={"wspace": 0.35}
)

# Make axes consistently indexable as a 2D array
axes = np.atleast_2d(axes)

for ax in axes.flat:
    ax.set_facecolor("black")
    ax.axis("off")

# Row 1, column 1: Lysosome channel with Fiji LUT
axes[0, 1].imshow(
    lys_img,
    cmap=fiji_cmap,
    vmin=0,
    vmax=3000
)
axes[0, 1].set_title(
    "Lysosome Image",
    color="black",
    fontsize=16,
    pad=12
)

# Row 1, column 2: Cell-body mask
cell_cmap = ListedColormap(["black", "#8A2BE2"])

axes[0, 2].imshow(
    cell_mask,
    cmap=cell_cmap
)
axes[0, 2].set_title(
    "Cell Body Mask",
    color="black",
    fontsize=16,
    pad=12
)

# Keep unused panels in the first row blank
axes[0, 0].axis("off")
axes[0, 3].axis("off")

# Binary colour maps for threshold masks
highlight_cmaps = [
    ListedColormap(["black", "cyan"]),
    ListedColormap(["black", "magenta"]),
    ListedColormap(["black", "yellow"]),
    ListedColormap(["black", "lime"])
]

# Row 2: thresholding methods
for idx, method in enumerate(valid_methods):
    ax = axes[1, idx]

    ax.imshow(
        masks[method],
        cmap=highlight_cmaps[idx % len(highlight_cmaps)]
    )

    ax.set_title(
        f"{method.capitalize()} Threshold\n"
        f"(Coverage: {row[f'{method}_coverage_pct']:.1f}%)",
        color="black",
        fontsize=16,
        pad=12
    )

# Hide unused threshold panels, if fewer than n_cols methods are available
for col in range(len(valid_methods), n_cols):
    axes[1, col].axis("off")

# Preserve editable text in the SVG
# plt.rcParams["svg.fonttype"] = "none"

plt.savefig(
    "Figures/threshold_comparison.svg",
    format="svg",
    bbox_inches="tight"
)

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ==============================================================================
# 1. PREPARE AND MERGE DATASETS
# ==============================================================================

# Extract human-readable string from label pairs
conditions_1 = [label[1] for label in labels[1]]
conditions_2 = [label[1] for label in labels[2]]

# Ensure the thresholding dataframes include the conditions
# Assumes rows in df1/df2 align with the indices processed from images_1/images_2
df1["Condition"] = [conditions_1[i] for i in df1["cell_id"]]
df2["Condition"] = [conditions_2[i] for i in df2["cell_id"]]

# Combine datasets for easier long-form melting
df1["Dataset"] = "Dataset 1"
df2["Dataset"] = "Dataset 2"
combined_df = pd.concat([df1, df2], ignore_index=True)

# Define methods and plotting orders
target_metrics = {
    "li_coverage_pct": "Li",
    "otsu_coverage_pct": "Otsu",
    "triangle_coverage_pct": "Triangle",
    "mean_coverage_pct": "Mean",
}

order_1 = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
order_2 = ["dox+\nPhTau-", "dox+\nPhTau+"]
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

# ==============================================================================
# 2. GENERATE 2x4 METHOD COMPARISON PLOT (Dataset 1 and Dataset 2 side-by-side)
# ==============================================================================

# 4 rows (methods), 2 columns (Dataset 1, Dataset 2)
fig, axes = plt.subplots(4, 2, figsize=(10, 12), sharey="row")

for row_idx, (metric_col, method_name) in enumerate(target_metrics.items()):
    # --- Dataset 1 Column ---
    ax_d1 = axes[row_idx, 0]
    sns.violinplot(
        ax=ax_d1,
        data=df1,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=order_1,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1,
    )
    ax_d1.set_ylabel(f"{method_name} Coverage (%)", fontsize=12)
    ax_d1.grid(axis="y", linestyle="--", alpha=0.5)
    ax_d1.set_xlabel("")

    # --- Dataset 2 Column ---
    ax_d2 = axes[row_idx, 1]
    sns.violinplot(
        ax=ax_d2,
        data=df2,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=order_2,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=1,
    )
    ax_d2.set_ylabel("")  # Hidden due to sharey="row"
    ax_d2.grid(axis="y", linestyle="--", alpha=0.5)
    ax_d2.set_xlabel("")

# Set Column Titles on the top row
axes[0, 0].set_title("Lysosome Coverage (Dataset 1)", fontsize=14, pad=10)
axes[0, 1].set_title("Lysosome Coverage (Dataset 2)", fontsize=14, pad=10)

# Set x-axis labels on the bottom row
for ax in axes[-1, :]:
    ax.set_xlabel("Condition", fontsize=12)

plt.show()

# ==============================================================================
# 3. GENERATE SEPARATE COMBINED PLOT FOR LI COVERAGE
# ==============================================================================

fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=combined_df,
    x="Condition",
    y="li_coverage_pct",
    hue="Condition",
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=1,
)
ax.set_title("Lysosome Coverage", fontsize=14, pad=10)
ax.set_ylabel("Li Coverage (%)", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.show()

# ==============================================================================
# 4. PRINT SUMMARY STATISTICS BY CONDITION
# ==============================================================================

for metric_col, method_name in target_metrics.items():
    print(f"\n============= {method_name.upper()} COVERAGE METRICS =============")
    print("\n[Dataset 1]")
    print(df1.groupby("Condition")[metric_col].describe().round(2))
    print("\n[Dataset 2]")
    print(df2.groupby("Condition")[metric_col].describe().round(2))
    print("\n[Combined]")
    print(combined_df.groupby("Condition")[metric_col].describe().round(2))

In [ ]:
# ==============================================================================
# STATISTICAL TESTS FOR ALL LYSOSOME-COVERAGE METHODS
# ==============================================================================

coverage_test_results = []

for metric_col, method_name in target_metrics.items():

    print(f"\n\n{'#' * 80}")
    print(f"{method_name} coverage: statistical analysis")
    print(f"{'#' * 80}")

    # Dataset 1
    result_d1 = run_group_tests(
        df=df1,
        value_col=metric_col,
        dataset_name="Dataset 1",
        metric_name=f"{method_name} coverage (%)",
        condition_order=order_1
    )

    if result_d1 is not None:
        coverage_test_results.append(result_d1)

    # Dataset 2
    result_d2 = run_group_tests(
        df=df2,
        value_col=metric_col,
        dataset_name="Dataset 2",
        metric_name=f"{method_name} coverage (%)",
        condition_order=order_2
    )

    if result_d2 is not None:
        coverage_test_results.append(result_d2)

    # Combined dataset
    result_combined = run_group_tests(
        df=combined_df,
        value_col=metric_col,
        dataset_name="Combined dataset",
        metric_name=f"{method_name} coverage (%)",
        condition_order=full_order
    )

    if result_combined is not None:
        coverage_test_results.append(result_combined)


# Combine all pairwise results into one table
if coverage_test_results:
    coverage_test_results = pd.concat(
        coverage_test_results,
        ignore_index=True
    )

    print("\n\nCombined pairwise coverage-test results:")
    print(coverage_test_results)
else:
    coverage_test_results = pd.DataFrame()
    print("No pairwise test results were generated.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_li, threshold_mean, threshold_otsu, threshold_triangle


# ==============================================================================
# 1. HELPER FUNCTION TO RECOMPUTE COVERAGE WITH GLOBAL THRESHOLDS
# ==============================================================================
def compute_global_coverage(images_list, labels_list, dataset_name, lysosome_channel):
    """Calculates one threshold per method for the entire dataset based on the lysosome channel,
    then computes individual cell coverage based on that static threshold.
    """
    pooled_lys_pixels = []
    cell_masks = []
    lys_images = []

    # --- STEP 1: EXTRACT VALID LYSOSOME PIXELS PER CELL ---
    for img in images_list:
        # Identify channel axis automatically
        channel_axis = np.argmin(img.shape)
        
        # Generate cell mask (filled solid body)
        raw_mask = np.any(img > 0, axis=channel_axis)
        cell_mask = binary_fill_holes(raw_mask)
        cell_masks.append(cell_mask)
        
        # Extract lysosome channel safely
        if channel_axis == 0:
            lys_img = img[lysosome_channel].astype(float)
        else:
            lys_img = img[:, :, lysosome_channel].astype(float)
        lys_images.append(lys_img)
        
        # Collect pixels inside the cell body for global thresholding
        lys_pixels = lys_img[cell_mask]
        if lys_pixels.size > 0:
            pooled_lys_pixels.append(lys_pixels)

    # Combine all valid lysosome pixels across the dataset
    all_non_zero_pixels = np.concatenate(pooled_lys_pixels)

    # --- STEP 2: CALCULATE GLOBAL THRESHOLDS ---
    global_thresholds = {
        "li_coverage_pct": threshold_li(all_non_zero_pixels),
        "otsu_coverage_pct": threshold_otsu(all_non_zero_pixels),
        "triangle_coverage_pct": threshold_triangle(all_non_zero_pixels),
        "mean_coverage_pct": threshold_mean(all_non_zero_pixels),
    }

    print(f"\n==== GLOBAL THRESHOLDS FOR {dataset_name.upper()} ====")
    for method, value in global_thresholds.items():
        print(f"{method.ljust(21)}: {value:.4f}")
    print("==========================================")

    # --- STEP 3: COMPUTE METRICS PER CELL USING GLOBAL THRESHOLDS ---
    rows = []
    for cell_id, (cell_mask, lys_img, label_pair) in enumerate(zip(cell_masks, lys_images, labels_list)):
        condition = label_pair[1]
        cell_area = np.sum(cell_mask)        

        cell_metrics = {
            "cell_id": cell_id, 
            "Condition": condition,
            "cell_area_px": cell_area
        }

        # Apply the fixed dataset-wide threshold to this specific cell
        for metric_col, thresh_value in global_thresholds.items():
            # Lysosome must exceed threshold AND stay within the cell mask
            lys_mask = (lys_img > thresh_value) & cell_mask
            coverage_pct = (np.sum(lys_mask) / cell_area) * 100
            cell_metrics[metric_col] = coverage_pct

        rows.append(cell_metrics)

    return pd.DataFrame(rows)

# ==============================================================================
# 2. PREPARE DATASETS WITH GLOBAL THRESHOLDS
# ==============================================================================
df1 = compute_global_coverage(images[1], labels[1], dataset_name="Dataset 1", lysosome_channel=0)
df2 = compute_global_coverage(images[2], labels[2], dataset_name="Dataset 2", lysosome_channel=0)

# Combine datasets for long-form tracking
df1["Dataset"] = "Dataset 1"
df2["Dataset"] = "Dataset 2"
combined_df = pd.concat([df1, df2], ignore_index=True)

In [ ]:
# Define methods and plotting orders
target_metrics = {
    "li_coverage_pct": "Li",
    "otsu_coverage_pct": "Otsu",
    "triangle_coverage_pct": "Triangle",
    "mean_coverage_pct": "Mean",
}
order_1 = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
order_2 = ["dox+\nPhTau-", "dox+\nPhTau+"]
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

# ==============================================================================
# 3. GENERATE 2x4 METHOD COMPARISON PLOT (Dataset 1 and Dataset 2 side-by-side)
# ==============================================================================
fig, axes = plt.subplots(4, 2, figsize=(10, 12), sharey="row")

for row_idx, (metric_col, method_name) in enumerate(target_metrics.items()):
    # --- Dataset 1 Column ---
    ax_d1 = axes[row_idx, 0]
    sns.violinplot(
        ax=ax_d1,
        data=df1,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=order_1,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=0,
    )
    ax_d1.set_ylabel(f"{method_name} Coverage (%)", fontsize=12)
    ax_d1.grid(axis="y", linestyle="--", alpha=0.5)
    ax_d1.set_xlabel("")

    # --- Dataset 2 Column ---
    ax_d2 = axes[row_idx, 1]
    sns.violinplot(
        ax=ax_d2,
        data=df2,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=order_2,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=0,
    )
    ax_d2.set_ylabel("")  # Hidden due to sharey="row"
    ax_d2.grid(axis="y", linestyle="--", alpha=0.5)
    ax_d2.set_xlabel("")

# Set Column Titles on the top row
axes[0, 0].set_title("Lysosome Coverage (Dataset 1)", fontsize=14, pad=10)
axes[0, 1].set_title("Lysosome Coverage (Dataset 2)", fontsize=14, pad=10)

# Set x-axis labels on the bottom row
for ax in axes[-1, :]:
    ax.set_xlabel("Condition", fontsize=12)

plt.show()

# ==============================================================================
# 4. GENERATE SEPARATE COMBINED PLOT FOR LI COVERAGE
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=combined_df,
    x="Condition",
    y="li_coverage_pct",
    hue="Condition",
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=0,
)
ax.set_title("Lysosome Coverage", fontsize=14, pad=10)
ax.set_ylabel("Li Coverage (%)", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.show()

# ==============================================================================
# 5. PRINT SUMMARY STATISTICS BY CONDITION
# ==============================================================================
for metric_col, method_name in target_metrics.items():
    print(f"\n============= {method_name.upper()} COVERAGE METRICS =============")
    print("\n[Dataset 1]")
    print(df1.groupby("Condition")[metric_col].describe().round(2))
    print("\n[Dataset 2]")
    print(df2.groupby("Condition")[metric_col].describe().round(2))
    print("\n[Combined]")
    print(combined_df.groupby("Condition")[metric_col].describe().round(2))

In [ ]:
# ==============================================================================
# 6. STATISTICAL TESTS FOR ALL COVERAGE METHODS
# ==============================================================================

coverage_test_results = []

for metric_col, method_name in target_metrics.items():

    print(f"\n\n{'#' * 80}")
    print(f"{method_name} coverage")
    print(f"{'#' * 80}")

    # Dataset 1
    result_d1 = run_group_tests(
        df=df1,
        value_col=metric_col,
        dataset_name="Dataset 1",
        metric_name=f"{method_name} coverage (%)",
        condition_order=order_1
    )

    if result_d1 is not None:
        coverage_test_results.append(result_d1)

    # Dataset 2
    result_d2 = run_group_tests(
        df=df2,
        value_col=metric_col,
        dataset_name="Dataset 2",
        metric_name=f"{method_name} coverage (%)",
        condition_order=order_2
    )

    if result_d2 is not None:
        coverage_test_results.append(result_d2)

    # Combined dataset
    result_combined = run_group_tests(
        df=combined_df,
        value_col=metric_col,
        dataset_name="Combined dataset",
        metric_name=f"{method_name} coverage (%)",
        condition_order=full_order
    )

    if result_combined is not None:
        coverage_test_results.append(result_combined)


# Combine all pairwise results
if coverage_test_results:
    coverage_test_results = pd.concat(
        coverage_test_results,
        ignore_index=True
    )

    print("\n\nAll pairwise coverage-test results:")
    print(coverage_test_results)
else:
    coverage_test_results = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_li, threshold_mean, threshold_otsu, threshold_triangle


# ==============================================================================
# 1. HELPER FUNCTION TO COMPUTE COMBINED GLOBAL THRESHOLDS
# ==============================================================================
def compute_combined_thresholds(images_list_1, images_list_2, lysosome_channel):
    """Calculates one threshold per method using valid lysosome pixels from BOTH datasets."""
    pooled_lys_pixels = []

    # Process and collect pixels for both datasets
    for img_list in [images_list_1, images_list_2]:
        for img in img_list:
            # Auto-detect channel axis orientation
            channel_axis = np.argmin(img.shape)
            
            # Create a filled solid mask of the cell body
            raw_mask = np.any(img > 0, axis=channel_axis)
            cell_mask = binary_fill_holes(raw_mask)
            
            # Extract lysosome channel safely
            if channel_axis == 0:
                lys_img = img[lysosome_channel].astype(float)
            else:
                lys_img = img[:, :, lysosome_channel].astype(float)
            
            # Gather pixels only within the cell boundaries
            lys_pixels = lys_img[cell_mask]
            if lys_pixels.size > 0:
                pooled_lys_pixels.append(lys_pixels)

    # Combine all valid lysosome pixels across both pools
    all_non_zero_pixels = np.concatenate(pooled_lys_pixels)

    global_thresholds = {
        "li_coverage_pct": threshold_li(all_non_zero_pixels),
        "otsu_coverage_pct": threshold_otsu(all_non_zero_pixels),
        "triangle_coverage_pct": threshold_triangle(all_non_zero_pixels),
        "mean_coverage_pct": threshold_mean(all_non_zero_pixels),
    }

    print("\n==== COMBINED GLOBAL THRESHOLDS (DATASET 1 & 2) ====")
    for method, value in global_thresholds.items():
        print(f"{method.ljust(21)}: {value:.4f}")
    print("====================================================")
    
    return global_thresholds


# ==============================================================================
# 2. HELPER FUNCTION TO APPLY THRESHOLDS TO A DATASET
# ==============================================================================
def apply_global_thresholds(images_list, labels_list, global_thresholds, lysosome_channel):
    """Computes individual cell coverage based on pre-calculated static thresholds."""
    rows = []
    for cell_id, (img, label_pair) in enumerate(zip(images_list, labels_list)):
        # Safely extract your condition column from the labels
        condition = label_pair[1]
        
        # Determine layout layout and isolate the cell area
        channel_axis = np.argmin(img.shape)
        raw_mask = np.any(img > 0, axis=channel_axis)
        cell_mask = binary_fill_holes(raw_mask)
        cell_area = np.sum(cell_mask)
        
        if channel_axis == 0:
            lys_img = img[lysosome_channel].astype(float)
        else:
            lys_img = img[:, :, lysosome_channel].astype(float)
        
        cell_metrics = {
            "cell_id": cell_id, 
            "Condition": condition, 
            "cell_area_px": cell_area
        }

        # Apply the shared global thresholds specifically inside the cell body mask
        for metric_col, thresh_value in global_thresholds.items():
            lys_mask = (lys_img > thresh_value) & cell_mask
            coverage_pct = (np.sum(lys_mask) / cell_area) * 100
            cell_metrics[metric_col] = coverage_pct

        rows.append(cell_metrics)

    return pd.DataFrame(rows)


# ==============================================================================
# 3. PREPARE DATASETS WITH COMBINED GLOBAL THRESHOLDS
# ==============================================================================
# Define the zero-indexed channel containing your lysosome data
LYSOSOME_CHANNEL_IDX = 0  # Adjust this index to your actual channel position

# Step A: Calculate thresholds once using both pools of images
shared_thresholds = compute_combined_thresholds(images[1], images[2], lysosome_channel=LYSOSOME_CHANNEL_IDX)

# Step B: Apply these exact same thresholds to both datasets
df1 = apply_global_thresholds(images[1], labels[1], shared_thresholds, lysosome_channel=LYSOSOME_CHANNEL_IDX)
df2 = apply_global_thresholds(images[2], labels[2], shared_thresholds, lysosome_channel=LYSOSOME_CHANNEL_IDX)

# Combine datasets for long-form tracking if needed later
df1["Dataset"] = "Dataset 1"
df2["Dataset"] = "Dataset 2"
combined_df = pd.concat([df1, df2], ignore_index=True)

# Define methods and plotting orders
target_metrics = {
    "li_coverage_pct": "Li",
    "otsu_coverage_pct": "Otsu",
    "triangle_coverage_pct": "Triangle",
    "mean_coverage_pct": "Mean",
}
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

In [ ]:
# ==============================================================================
# 4. GENERATE 2-COLUMN PLOT (DATASET 1 & DATASET 2) AND SAVE
# ==============================================================================
fig, axes = plt.subplots(4, 2, figsize=(10, 12), sharex="col", sharey="row")

for row_idx, (metric_col, method_name) in enumerate(target_metrics.items()):
    # --- Dataset 1 Column ---
    ax_d1 = axes[row_idx, 0]
    sns.violinplot(
        ax=ax_d1,
        data=df1,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=full_order,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=0,
    )
    ax_d1.set_ylabel(f"{method_name}\nCoverage (%)", fontsize=12)

    # --- Dataset 2 Column ---
    ax_d2 = axes[row_idx, 1]
    sns.violinplot(
        ax=ax_d2,
        data=df2,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=["dox+\nPhTau-", "dox+\nPhTau+"],
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=0,
    )
    ax_d2.set_ylabel("")

    ax_d1.grid(axis="y", linestyle="--", alpha=0.5)
    ax_d2.grid(axis="y", linestyle="--", alpha=0.5)

axes[0, 0].set_title("Lysosome Coverage (Dataset 1)", fontsize=14, pad=10)
axes[0, 1].set_title("Lysosome Coverage (Dataset 2)", fontsize=14, pad=10)

for ax in axes.flat:
    ax.set_xlabel("")
for ax in axes[-1, :]:
    ax.set_xlabel("Condition", fontsize=12)

plt.savefig("Figures/lysosome_coverage_by_method.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 5. GENERATE AND SAVE COMBINED PLOT INDEPENDENTLY (LI THRESHOLDING ONLY)
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=combined_df,
    x="Condition",
    y="li_coverage_pct",
    hue="Condition",
    order=full_order,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=0,
)
ax.set_title("Lysosome Coverage", fontsize=14, pad=10)
ax.set_ylabel("Li Coverage (%)", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.savefig("Figures/lysosome_coverage_combined.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 6. PRINT SUMMARY STATISTICS BY CONDITION
# ==============================================================================
for metric_col, method_name in target_metrics.items():
    print(f"\n============= {method_name.upper()} COVERAGE METRICS =============")
    print("\n[Dataset 1]")
    print(df1.groupby("Condition")[metric_col].describe().round(2))
    print("\n[Dataset 2]")
    print(df2.groupby("Condition")[metric_col].describe().round(2))
    print("\n[Combined Dataset]")
    print(combined_df.groupby("Condition")[metric_col].describe().round(2))

In [ ]:
# ==============================================================================
# 7. STATISTICAL TESTS FOR ALL COVERAGE METRICS
# ==============================================================================

coverage_test_results = []

for metric_col, method_name in target_metrics.items():

    print(f"\n\n{'#' * 80}")
    print(f"{method_name} coverage")
    print(f"{'#' * 80}")

    # Dataset 1
    result_d1 = run_group_tests(
        df=df1,
        value_col=metric_col,
        dataset_name="Dataset 1",
        metric_name=f"{method_name} coverage (%)",
        condition_order=full_order
    )

    if result_d1 is not None:
        coverage_test_results.append(result_d1)

    # Dataset 2
    result_d2 = run_group_tests(
        df=df2,
        value_col=metric_col,
        dataset_name="Dataset 2",
        metric_name=f"{method_name} coverage (%)",
        condition_order=full_order
    )

    if result_d2 is not None:
        coverage_test_results.append(result_d2)

    # Combined dataset
    result_combined = run_group_tests(
        df=combined_df,
        value_col=metric_col,
        dataset_name="Combined dataset",
        metric_name=f"{method_name} coverage (%)",
        condition_order=full_order
    )

    if result_combined is not None:
        coverage_test_results.append(result_combined)


# Store all pairwise results in one DataFrame
if coverage_test_results:
    coverage_test_results = pd.concat(
        coverage_test_results,
        ignore_index=True
    )

    print("\n\nAll pairwise coverage-test results:")
    print(coverage_test_results)
else:
    coverage_test_results = pd.DataFrame()
    print("No pairwise results were generated.")

### Lysosome size

In [ ]:
import numpy as np
import pandas as pd
from scipy.ndimage import binary_fill_holes


# ==============================================================================
# 1. PROCESS DATASETS AND COMPUTE AVERAGE LYSOSOME SIZE
# ==============================================================================
def process_dataset_with_lysosomes(
    images_list, labels_list, lysosomes_list, global_thresholds, lysosome_channel
):
    """Processes clean datasets to compute metrics and average lysosome size,

    accounting for multi-channel image layouts.
    """
    rows = []

    for cell_id, (img, label_pair, lys_count) in enumerate(
        zip(images_list, labels_list, lysosomes_list)
    ):
        # Extract condition safely
        condition = label_pair
        
        # Skip rows missing lysosome data if any exist
        if pd.isna(lys_count):
            continue

        # Auto-detect channel axis orientation
        channel_axis = np.argmin(img.shape)
        
        # Create a filled solid mask of the cell body
        raw_mask = np.any(img > 0, axis=channel_axis)
        cell_mask = binary_fill_holes(raw_mask)
        cell_area = np.sum(cell_mask)

        # Extract lysosome channel safely
        if channel_axis == 0:
            lys_img = img[lysosome_channel].astype(float)
        else:
            lys_img = img[:, :, lysosome_channel].astype(float)

        cell_metrics = {
            "cell_id": cell_id,
            "Condition": condition,
            "Lysosome_Count": lys_count,
            "Total_Pixels": cell_area,  # Represents true solid cell area now
        }

        # Calculate coverage percentages and back-calculate pixel areas
        for metric_col, thresh_value in global_thresholds.items():
            # Lysosome must exceed threshold AND stay within the cell mask
            lys_mask = (lys_img > thresh_value) & cell_mask
            mask_pixel_area = np.sum(lys_mask)

            # 1. Store percentage coverage based on true cell area
            coverage_pct = (mask_pixel_area / cell_area) * 100
            cell_metrics[metric_col] = coverage_pct

            # 2. Compute Average Lysosome Size (Area / Count)
            method_prefix = metric_col.split("_")[0]
            if lys_count > 0:
                cell_metrics[f"{method_prefix}_avg_size"] = (
                    mask_pixel_area / lys_count
                )
            else:
                cell_metrics[f"{method_prefix}_avg_size"] = 0.0

        rows.append(cell_metrics)

    return pd.DataFrame(rows)


# ==============================================================================
# 2. RUN PIPELINE
# ==============================================================================

# Process both sets using the shared_thresholds calculated earlier
df1_scaled = process_dataset_with_lysosomes(
    images[1], labels[1], lysosomes_1, shared_thresholds, lysosome_channel=0
)
df2_scaled = process_dataset_with_lysosomes(
    images[2], labels[2], lysosomes_2, shared_thresholds, lysosome_channel=0
)

# Tag and pool for the combined analysis
df1_scaled["Dataset"] = "Dataset 1"
df2_scaled["Dataset"] = "Dataset 2"
combined_scaled_df = pd.concat([df1_scaled, df2_scaled], ignore_index=True)

In [ ]:
import ast
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# ==============================================================================
# CONFIGURATION: Choose which computed metric you want to plot
# ==============================================================================
method_prefix = list(shared_thresholds.keys())[0].split("_")[0]
target_metric = f"{method_prefix}_avg_size"

# Pixel-to-micrometer conversion (assuming avg_size is an area in pixels)
pixel_width = 0.0739859
area_conversion = pixel_width ** 2


# ==============================================================================
# Helper Function to Safely Parse String-ified Lists
# ==============================================================================
def extract_human_label(label):
    # If it's already a list or tuple, extract index 1
    if isinstance(label, (list, tuple)):
        return label[1] if len(label) > 1 else label[0]

    # If it's a string, attempt to parse it as a literal Python object
    if isinstance(label, str):
        label = label.strip()
        if label.startswith("[") or label.startswith("("):
            try:
                parsed = ast.literal_eval(label)
                if isinstance(parsed, (list, tuple)):
                    return parsed[1] if len(parsed) > 1 else parsed[0]
            except (ValueError, SyntaxError):
                pass  # Fallback to returning the raw string if parsing fails

    return label


# ==============================================================================
# 3. PREPARE PLOTTING DATAFRAMES (Converted to µm²)
# ==============================================================================
df1_plot = pd.DataFrame(
    {
        target_metric: df1_scaled[target_metric] * area_conversion,
        "Condition": [
            extract_human_label(label) for label in df1_scaled["Condition"]
        ],
    }
).dropna()

df2_plot = pd.DataFrame(
    {
        target_metric: df2_scaled[target_metric] * area_conversion,
        "Condition": [
            extract_human_label(label) for label in df2_scaled["Condition"]
        ],
    }
).dropna()

# Combine both datasets using the filtered plotting dataframes
df_combined_plot = pd.concat([df1_plot, df2_plot], ignore_index=True)

# Fixed alignment orders
order_1 = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
order_2 = ["dox+\nPhTau-", "dox+\nPhTau+"]
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

# Format metric label for the y-axis (e.g., "Method Avg Size (µm²)")
formatted_ylabel = f"{target_metric.replace('_', ' ').title()} (µm²)"

# ==============================================================================
# 4. GENERATE 2-COLUMN PLOT (DATASET 1 & DATASET 2) AND SAVE
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

datasets = [
    (df1_plot, order_1, 'Lysosome Size (Dataset 1)', axes[0]), 
    (df2_plot, order_2, 'Lysosome Size (Dataset 2)', axes[1])
]

for df_subset, order_subset, title_text, ax in datasets:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y=target_metric,
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=0,
    )
    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel(formatted_ylabel, fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig(f"Figures/{target_metric}_by_dataset.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 5. GENERATE AND SAVE COMBINED PLOT INDEPENDENTLY
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined_plot,
    x="Condition",
    y=target_metric,
    hue="Condition",
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=0,
)
ax.set_title("Lysosome Size", fontsize=14, pad=10)
ax.set_ylabel(formatted_ylabel, fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.xticks(rotation=0)

plt.savefig(f"Figures/{target_metric}_combined.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 6. SUMMARY STATISTICS (Rounded to 2 decimal places for µm² precision)
# ==============================================================================
print(f"--- Dataset 1 Summary Statistics ({target_metric} in µm²) ---")
print(df1_plot.groupby("Condition")[target_metric].describe().round(2))

print(f"\n--- Dataset 2 Summary Statistics ({target_metric} in µm²) ---")
print(df2_plot.groupby("Condition")[target_metric].describe().round(2))

print(f"\n--- Combined Dataset Summary Statistics ({target_metric} in µm²) ---")
print(df_combined_plot.groupby("Condition")[target_metric].describe().round(2))

In [ ]:
# ==============================================================================
# 7. STATISTICAL TESTS FOR LYSOSOME SIZE
# ==============================================================================

size_test_results = []

# Dataset 1
result_size_d1 = run_group_tests(
    df=df1_plot,
    value_col=target_metric,
    dataset_name="Dataset 1",
    metric_name=f"{target_metric.replace('_', ' ').title()} (µm²)",
    condition_order=full_order
)

if result_size_d1 is not None:
    size_test_results.append(result_size_d1)


# Dataset 2
result_size_d2 = run_group_tests(
    df=df2_plot,
    value_col=target_metric,
    dataset_name="Dataset 2",
    metric_name=f"{target_metric.replace('_', ' ').title()} (µm²)",
    condition_order=full_order
)

if result_size_d2 is not None:
    size_test_results.append(result_size_d2)


# Combined dataset
result_size_combined = run_group_tests(
    df=df_combined_plot,
    value_col=target_metric,
    dataset_name="Combined dataset",
    metric_name=f"{target_metric.replace('_', ' ').title()} (µm²)",
    condition_order=full_order
)

if result_size_combined is not None:
    size_test_results.append(result_size_combined)


# Combine all pairwise results into one DataFrame
if size_test_results:
    size_test_results = pd.concat(
        size_test_results,
        ignore_index=True
    )

    print("\n\nAll lysosome-size pairwise test results:")
    print(size_test_results)
else:
    size_test_results = pd.DataFrame()
    print("No pairwise results were generated.")

### Lysosome spot intensity

In [ ]:
import numpy as np

def extract_spot_intensities(image_stack, coordinates, coord_order="xy", 
                             pixel_size_um=0.0739859, radius_um=0.56, 
                             aggregation="mean"):
    """
    Extract regional pixel intensities at spot coordinates, accounting for lysosome size.
    """
    radius_pix = max(1, int(np.round(radius_um / pixel_size_um)))
    spot_intensities = []

    for cell_idx, cell_coords in enumerate(coordinates):
        cell_image = image_stack[cell_idx]
        cell_coords = np.asarray(cell_coords)

        if len(cell_coords) == 0:
            # Always 2D output now since we extract only channel 0
            spot_intensities.append(np.empty(0))
            continue

        cell_coords = np.rint(cell_coords).astype(int)

        if coord_order == "xy":
            x = cell_coords[:, 0]
            y = cell_coords[:, 1]
        elif coord_order == "yx":
            y = cell_coords[:, 0]
            x = cell_coords[:, 1]
        else:
            raise ValueError("coord_order must be 'xy' or 'yx'")

        height, width = cell_image.shape[:2]
        intensities = []

        for xi, yi in zip(x, y):
            ymin = max(0, yi - radius_pix)
            ymax = min(height, yi + radius_pix + 1)
            xmin = max(0, xi - radius_pix)
            xmax = min(width, xi + radius_pix + 1)

            if ymin >= ymax or xmin >= xmax:
                continue

            # Extract local region from channel 0 only
            if cell_image.ndim == 3:
                local_patch = cell_image[ymin:ymax, xmin:xmax, 0]
            else:
                local_patch = cell_image[ymin:ymax, xmin:xmax]
            
            # Aggregate over the 2D spatial patch (axis=(0,1) is no longer needed since it's strictly 2D)
            if aggregation == "mean":
                spot_val = np.mean(local_patch)
            elif aggregation == "sum":
                spot_val = np.sum(local_patch)
            elif aggregation == "max":
                spot_val = np.max(local_patch)
            else:
                raise ValueError("aggregation must be 'mean', 'sum', or 'max'")

            intensities.append(spot_val)

        spot_intensities.append(np.array(intensities))

    return spot_intensities

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ==========================================
# 1. EXTRACT SPOT INTENSITIES PER CELL
# ==========================================
# Assuming 'images', 'coord_map', and 'labels' are already populated from your first snippet
spot_intensities_1 = extract_spot_intensities(images[1], coord_map[1], coord_order="xy")
spot_intensities_2 = extract_spot_intensities(images[2], coord_map[2], coord_order="xy")

# Helper function to compute a metric per cell (e.g., mean intensity of spots, 
# or if multichannel, pick a specific channel like channel 0 or mean across channels)
def get_mean_spot_intensity(spot_list):
    mean_intensities = []
    for cell_spots in spot_list:
        if len(cell_spots) == 0:
            mean_intensities.append(np.nan)
        else:
            # If multichannel (2D array: n_spots, n_channels), let's take channel 0 
            # (modify slice/index if you want a different channel or total mean)
            if cell_spots.ndim == 2:
                values = cell_spots[:, 0] 
            else:
                values = cell_spots
            mean_intensities.append(np.mean(values))
    return mean_intensities

intensities_1 = get_mean_spot_intensity(spot_intensities_1)
intensities_2 = get_mean_spot_intensity(spot_intensities_2)

# ==========================================
# 2. BUILD DATAFRAMES
# ==========================================
df1_int = pd.DataFrame(
    {
        "Mean_Spot_Intensity": intensities_1,
        "Condition": [label[1] for label in labels[1]],
    }
).dropna()

df2_int = pd.DataFrame(
    {
        "Mean_Spot_Intensity": intensities_2,
        "Condition": [label[1] for label in labels[2]],
    }
).dropna()

# Combine both datasets
df_combined_int = pd.concat([df1_int, df2_int], ignore_index=True)

# Define orders to maintain colour alignment across all plots
order_1 = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]
order_2 = ["dox+\nPhTau-", "dox+\nPhTau+"]
full_order = ["dox-", "dox+\nPhTau-", "dox+\nPhTau+"]

# ==========================================
# 3. GENERATE 2-COLUMN PLOT (DATASET 1 & DATASET 2) AND SAVE
# ==========================================
fig, axes = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

datasets = [
    (df1_int, order_1, 'Lysosome Intensity (Dataset 1)', axes[0]), 
    (df2_int, order_2, 'Lysosome Intensity (Dataset 2)', axes[1])
]

for df_subset, order_subset, title_text, ax in datasets:
    sns.violinplot(
        ax=ax,
        data=df_subset,
        x="Condition",
        y="Mean_Spot_Intensity",
        hue="Condition",
        order=order_subset,
        hue_order=full_order,
        palette=traffic_light_map,
        legend=False,
        cut=0,
    )
    ax.set_title(title_text, fontsize=14, pad=10)
    ax.set_ylabel("Mean Intensity per Cell", fontsize=12)
    ax.set_xlabel("Condition", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
    plt.sca(ax)
    plt.xticks(rotation=0)

plt.savefig("Figures/mean_spot_intensity_by_dataset.svg", format="svg", bbox_inches="tight")
plt.show()

# ==========================================
# 4. GENERATE AND SAVE COMBINED PLOT INDEPENDENTLY
# ==========================================
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_combined_int,
    x="Condition",
    y="Mean_Spot_Intensity",
    hue="Condition",
    order=order_1,
    hue_order=full_order,
    palette=traffic_light_map,
    legend=False,
    cut=0,
)
ax.set_title("Lysosome Intensity", fontsize=14, pad=10)
ax.set_ylabel("Mean Intensity per Cell", fontsize=12)
ax.set_xlabel("Condition", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)
ax.ticklabel_format(style='sci', axis='y', scilimits=(0, 0))
plt.xticks(rotation=0)

plt.savefig("Figures/mean_spot_intensity_combined.svg", format="svg", bbox_inches="tight")
plt.show()

# ==========================================
# 5. DISPLAY SUMMARY STATISTICS
# ==========================================
print("--- Dataset 1 Intensity Summary Statistics ---")
print(df1_int.groupby("Condition")["Mean_Spot_Intensity"].describe().round(1))

print("\n--- Dataset 2 Intensity Summary Statistics ---")
print(df2_int.groupby("Condition")["Mean_Spot_Intensity"].describe().round(1))

print("\n--- Combined Dataset Intensity Summary Statistics ---")
print(df_combined_int.groupby("Condition")["Mean_Spot_Intensity"].describe().round(1))

In [ ]:
# ==============================================================================
# 6. STATISTICAL TESTS FOR MEAN SPOT INTENSITY
# ==============================================================================

intensity_test_results = []

# Dataset 1
result_int_d1 = run_group_tests(
    df=df1_int,
    value_col="Mean_Spot_Intensity",
    dataset_name="Dataset 1",
    metric_name="Mean spot intensity",
    condition_order=full_order
)

if result_int_d1 is not None:
    intensity_test_results.append(result_int_d1)


# Dataset 2
result_int_d2 = run_group_tests(
    df=df2_int,
    value_col="Mean_Spot_Intensity",
    dataset_name="Dataset 2",
    metric_name="Mean spot intensity",
    condition_order=full_order
)

if result_int_d2 is not None:
    intensity_test_results.append(result_int_d2)


# Combined dataset
result_int_combined = run_group_tests(
    df=df_combined_int,
    value_col="Mean_Spot_Intensity",
    dataset_name="Combined dataset",
    metric_name="Mean spot intensity",
    condition_order=full_order
)

if result_int_combined is not None:
    intensity_test_results.append(result_int_combined)


# Combine all pairwise results into a single DataFrame
if intensity_test_results:
    intensity_test_results = pd.concat(
        intensity_test_results,
        ignore_index=True
    )

    print("\n\nAll mean-spot-intensity pairwise test results:")
    print(intensity_test_results)
else:
    intensity_test_results = pd.DataFrame()
    print("No pairwise results were generated.")

### Correlation with Tau metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import binary_fill_holes
from skimage.filters import threshold_li, threshold_mean, threshold_otsu, threshold_triangle

# ==============================================================================
# 1. HELPER FUNCTION TO COMPUTE COMBINED GLOBAL THRESHOLDS
# ==============================================================================
def compute_combined_thresholds(images_list_1, images_list_2, lysosome_channel):
    """Calculates one threshold per method using valid lysosome pixels from BOTH datasets."""
    pooled_lys_pixels = []

    for img_list in [images_list_1, images_list_2]:
        for img in img_list:
            channel_axis = np.argmin(img.shape)
            raw_mask = np.any(img > 0, axis=channel_axis)
            cell_mask = binary_fill_holes(raw_mask)
            
            if channel_axis == 0:
                lys_img = img[lysosome_channel].astype(float)
            else:
                lys_img = img[:, :, lysosome_channel].astype(float)
            
            lys_pixels = lys_img[cell_mask]
            if lys_pixels.size > 0:
                pooled_lys_pixels.append(lys_pixels)

    all_non_zero_pixels = np.concatenate(pooled_lys_pixels)

    global_thresholds = {
        "li_coverage_pct": threshold_li(all_non_zero_pixels),
        "otsu_coverage_pct": threshold_otsu(all_non_zero_pixels),
        "triangle_coverage_pct": threshold_triangle(all_non_zero_pixels),
        "mean_coverage_pct": threshold_mean(all_non_zero_pixels),
    }

    print("\n==== COMBINED GLOBAL THRESHOLDS (DATASET 1 & 2) ====")
    for method, value in global_thresholds.items():
        print(f"{method.ljust(21)}: {value:.4f}")
    print("====================================================")
    
    return global_thresholds


# ==============================================================================
# 2. UPDATED HELPER FUNCTION TO APPLY THRESHOLDS AND EXTRACT PHTAU METRICS
# ==============================================================================
def apply_global_thresholds_with_phtau(images_list, labels_list, global_thresholds, lysosome_channel, phtau_channel):
    """Computes individual cell coverage and extracts PhTau channel metrics within the cell body mask."""
    rows = []
    for cell_id, (img, label_pair) in enumerate(zip(images_list, labels_list)):
        condition = label_pair[1]
        
        channel_axis = np.argmin(img.shape)
        raw_mask = np.any(img > 0, axis=channel_axis)
        cell_mask = binary_fill_holes(raw_mask)
        cell_area = np.sum(cell_mask)
        
        if channel_axis == 0:
            lys_img = img[lysosome_channel].astype(float)
            phtau_img = img[phtau_channel].astype(float)
        else:
            lys_img = img[:, :, lysosome_channel].astype(float)
            phtau_img = img[:, :, phtau_channel].astype(float)
        
        cell_metrics = {
            "cell_id": cell_id, 
            "Condition": condition, 
            "cell_area_px": cell_area
        }

        # 1. Calculate Lysosome Coverage Metrics
        for metric_col, thresh_value in global_thresholds.items():
            lys_mask = (lys_img > thresh_value) & cell_mask
            coverage_pct = (np.sum(lys_mask) / cell_area) * 100
            cell_metrics[metric_col] = coverage_pct

        # 2. Extract PhTau Metrics strictly within the cell body boundary
        phtau_pixels = phtau_img[cell_mask]
        cell_metrics["phtau_sum"] = np.sum(phtau_pixels)
        cell_metrics["phtau_mean"] = np.mean(phtau_pixels)
        cell_metrics["phtau_std"] = np.std(phtau_pixels)

        rows.append(cell_metrics)

    return pd.DataFrame(rows)


# ==============================================================================
# 3. PREPARE DATASETS WITH NEW CHANNEL MAPPINGS
# ==============================================================================
LYSOSOME_CHANNEL_IDX = 0  # Adjust as needed
PHTAU_CHANNEL_IDX = 1     # Zero-indexed position of your PhTau channel

# Step A: Shared thresholds calculated identically
shared_thresholds = compute_combined_thresholds(images[1], images[2], lysosome_channel=LYSOSOME_CHANNEL_IDX)

# Step B: Apply thresholds and extract PhTau metrics simultaneously
df1 = apply_global_thresholds_with_phtau(
    images[1], labels[1], shared_thresholds, lysosome_channel=LYSOSOME_CHANNEL_IDX, phtau_channel=PHTAU_CHANNEL_IDX
)
df2 = apply_global_thresholds_with_phtau(
    images[2], labels[2], shared_thresholds, lysosome_channel=LYSOSOME_CHANNEL_IDX, phtau_channel=PHTAU_CHANNEL_IDX
)

df1["Dataset"] = "Dataset 1"
df2["Dataset"] = "Dataset 2"
combined_df = pd.concat([df1, df2], ignore_index=True)

# Define groupings
target_metrics = ["li_coverage_pct", "otsu_coverage_pct", "triangle_coverage_pct", "mean_coverage_pct"]
phtau_metrics = ["phtau_sum", "phtau_mean", "phtau_std"]


# ==============================================================================
# 4. GENERATE CORRELATION PLOTS (Using Traffic Light Palette)
# ==============================================================================
traffic_light_map = {
    "dox-": "#2ecc71",           # Green
    "dox+\nPhTau-": "#f1c40f",   # Yellow
    "dox+\nPhTau+": "#e74c3c",   # Red
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Dynamically gather unique conditions present across both dataframes
unique_conditions = pd.concat([df1["Condition"], df2["Condition"]]).unique()

active_palette = {
    k: v for k, v in traffic_light_map.items() if k in unique_conditions
}

# FIX: Use [df1, df2] instead of [df1_counts, df2_counts]
for idx, (df_target, name) in enumerate(
    zip([df1, df2], ["Dataset 1", "Dataset 2"])
):
    sns.scatterplot(
        ax=axes[idx],
        data=df_target,
        x="li_coverage_pct",
        y="phtau_mean",
        hue="Condition",
        palette=active_palette,
        hue_order=[
            k for k in traffic_light_map.keys() if k in unique_conditions
        ],
        alpha=0.8,
    )
    axes[idx].set_title(f"{name}: PhTau Intensity vs. Li Lysosome Coverage")
    axes[idx].set_xlabel("Li Lysosome Coverage (%)")
    if idx == 0:
        axes[idx].set_ylabel("PhTau Mean Intensity (per cell)")
    else:
        axes[idx].set_ylabel("")

plt.tight_layout()
plt.show()


# ==============================================================================
# 5. PRINT CORRELATION MATRIX PER DATASET
# ==============================================================================
for name, df_target in zip(["Dataset 1", "Dataset 2"], [df1, df2]):
    print(f"\n============= {name.upper()} CORRELATION MATRIX =============")
    # Select columns to evaluate
    analysis_cols = target_metrics + phtau_metrics
    corr_matrix = df_target[analysis_cols].corr(method="pearson")
    
    # Isolate specifically the relationship between Lysosome metrics and PhTau metrics
    print(corr_matrix.loc[target_metrics, phtau_metrics].round(3))

In [ ]:
# ==============================================================================
# 1. EXTRACT LYSOSOME SPOT COUNTS
# ==============================================================================
# Assuming `coord_map` is already defined in your workspace
lysosomes_1 = extract_lysosome_counts(coord_map.get(1, []))
lysosomes_2 = extract_lysosome_counts(coord_map.get(2, []))

print(f"Dataset 1: Extracted {len(lysosomes_1)} frame spot counts.")
print(f"Dataset 2: Extracted {len(lysosomes_2)} frame spot counts.")


# ==============================================================================
# 2. HELPER FUNCTION TO EXTRACT PHTAU & ADD COUNTS
# ==============================================================================
def extract_phtau_with_counts(images_list, labels_list, spot_counts_list):
    """Extracts cell area and PhTau metrics, appending pre-calculated spot counts."""
    rows = []
    for cell_id, (img, label_pair, lys_count) in enumerate(
        zip(images_list, labels_list, spot_counts_list)
    ):
        condition = label_pair[1]

        channel_axis = np.argmin(img.shape)
        raw_mask = np.any(img > 0, axis=channel_axis)
        cell_mask = binary_fill_holes(raw_mask)
        cell_area = np.sum(cell_mask)

        if channel_axis == 0:
            phtau_img = img[PHTAU_CHANNEL_IDX].astype(float)
        else:
            phtau_img = img[:, :, PHTAU_CHANNEL_IDX].astype(float)

        cell_metrics = {
            "cell_id": cell_id,
            "Condition": condition,
            "cell_area_px": cell_area,
            "lysosome_count": lys_count,  # Replaces coverage metrics
        }

        phtau_pixels = phtau_img[cell_mask]
        cell_metrics["phtau_sum"] = np.sum(phtau_pixels)
        cell_metrics["phtau_mean"] = np.mean(phtau_pixels)
        cell_metrics["phtau_std"] = np.std(phtau_pixels)

        rows.append(cell_metrics)

    return pd.DataFrame(rows)


# ==============================================================================
# 3. BUILD COMBINED DATAFRAME
# ==============================================================================
df1_counts = extract_phtau_with_counts(images[1], labels[1], lysosomes_1)
df2_counts = extract_phtau_with_counts(images[2], labels[2], lysosomes_2)

df1_counts["Dataset"] = "Dataset 1"
df2_counts["Dataset"] = "Dataset 2"
combined_counts_df = pd.concat([df1_counts, df2_counts], ignore_index=True)

target_metrics = ["lysosome_count"]
phtau_metrics = ["phtau_sum", "phtau_mean", "phtau_std"]


# ==============================================================================
# 4. GENERATE CORRELATION PLOTS (Using Traffic Light Palette)
# ==============================================================================
traffic_light_map = {
    "dox-": "#2ecc71",  # Green
    "dox+\nPhTau-": "#f1c40f",  # Yellow
    "dox+\nPhTau+": "#e74c3c",  # Red
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

# Filter the palette to match only the conditions present in your data,
# falling back gracefully if a category isn't in a specific subset.
active_palette = {
    k: v for k, v in traffic_light_map.items() if k in unique_conditions
}

for idx, (df_target, name) in enumerate(
    zip([df1_counts, df2_counts], ["Dataset 1", "Dataset 2"])
):
    sns.scatterplot(
        ax=axes[idx],
        data=df_target,
        x="lysosome_count",
        y="phtau_mean",
        hue="Condition",
        palette=active_palette,
        hue_order=[
            k for k in traffic_light_map.keys() if k in unique_conditions
        ],
        alpha=0.8,
    )
    axes[idx].set_title(f"{name}: PhTau Intensity vs. Lysosome Count")
    axes[idx].set_xlabel("Lysosome Count")
    if idx == 0:
        axes[idx].set_ylabel("PhTau Mean Intensity (per cell)")
    else:
        axes[idx].set_ylabel("")

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. PRINT CORRELATION MATRIX PER DATASET
# ==============================================================================
for name, df_target in zip(["Dataset 1", "Dataset 2"], [df1_counts, df2_counts]):
    print(f"\n============= {name.upper()} CORRELATION MATRIX (COUNTS) =============")
    analysis_cols = target_metrics + phtau_metrics
    corr_matrix = df_target[analysis_cols].corr(method="pearson")
    print(corr_matrix.loc[target_metrics, phtau_metrics].round(3))

In [ ]:
# ==============================================================================
# 1. EXTRACT LYSOSOME SPOT INTENSITY METRICS
# ==============================================================================
# Assuming `spot_intensities_1` and `spot_intensities_2` are already defined in your workspace
# (e.g., lists or arrays of spot intensities per cell/frame)

print(f"Dataset 1: Loaded {len(spot_intensities_1)} cell intensity arrays.")
print(f"Dataset 2: Loaded {len(spot_intensities_2)} cell intensity arrays.")


# ==============================================================================
# 2. HELPER FUNCTION TO EXTRACT PHTAU & ADD INTENSITY METRICS
# ==============================================================================
def extract_phtau_with_intensities(images_list, labels_list, intensities_list):
    """Extracts cell area and PhTau metrics, appending summary statistics

    of spot intensities (sum, mean, std).
    """
    rows = []
    for cell_id, (img, label_pair, spot_ints) in enumerate(
        zip(images_list, labels_list, intensities_list)
    ):
        condition = label_pair[1]

        channel_axis = np.argmin(img.shape)
        raw_mask = np.any(img > 0, axis=channel_axis)
        cell_mask = binary_fill_holes(raw_mask)
        cell_area = np.sum(cell_mask)

        if channel_axis == 0:
            phtau_img = img[PHTAU_CHANNEL_IDX].astype(float)
        else:
            phtau_img = img[:, :, PHTAU_CHANNEL_IDX].astype(float)

        # Handle potential empty spot intensity arrays gracefully
        spot_ints_arr = np.array(spot_ints)
        if spot_ints_arr.size > 0:
            spot_sum = np.sum(spot_ints_arr)
            spot_mean = np.mean(spot_ints_arr)
            spot_std = np.std(spot_ints_arr)
        else:
            spot_sum, spot_mean, spot_std = 0.0, 0.0, 0.0

        cell_metrics = {
            "cell_id": cell_id,
            "Condition": condition,
            "cell_area_px": cell_area,
            "spot_intensity_sum": spot_sum,
            "spot_intensity_mean": spot_mean,
            "spot_intensity_std": spot_std,
        }

        phtau_pixels = phtau_img[cell_mask]
        cell_metrics["phtau_sum"] = np.sum(phtau_pixels)
        cell_metrics["phtau_mean"] = np.mean(phtau_pixels)
        cell_metrics["phtau_std"] = np.std(phtau_pixels)

        rows.append(cell_metrics)

    return pd.DataFrame(rows)


# ==============================================================================
# 3. BUILD COMBINED DATAFRAME
# ==============================================================================
df1_ints = extract_phtau_with_intensities(
    images[1], labels[1], spot_intensities_1
)
df2_ints = extract_phtau_with_intensities(
    images[2], labels[2], spot_intensities_2
)

df1_ints["Dataset"] = "Dataset 1"
df2_ints["Dataset"] = "Dataset 2"
combined_ints_df = pd.concat([df1_ints, df2_ints], ignore_index=True)

target_metrics = [
    "spot_intensity_sum",
    "spot_intensity_mean",
    "spot_intensity_std",
]
phtau_metrics = ["phtau_sum", "phtau_mean", "phtau_std"]


# ==============================================================================
# 4. GENERATE CORRELATION PLOTS (Using Traffic Light Palette)
# ==============================================================================
traffic_light_map = {
    "dox-": "#2ecc71",  # Green
    "dox+\nPhTau-": "#f1c40f",  # Yellow
    "dox+\nPhTau+": "#e74c3c",  # Red
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

active_palette = {
    k: v for k, v in traffic_light_map.items() if k in unique_conditions
}

for idx, (df_target, name) in enumerate(
    zip([df1_ints, df2_ints], ["Dataset 1", "Dataset 2"])
):
    sns.scatterplot(
        ax=axes[idx],
        data=df_target,
        x="spot_intensity_mean",  # Comparing mean spot intensity vs PhTau mean
        y="phtau_mean",
        hue="Condition",
        palette=active_palette,
        hue_order=[
            k for k in traffic_light_map.keys() if k in unique_conditions
        ],
        alpha=0.8,
    )
    axes[idx].set_title(f"{name}: PhTau Intensity vs. Mean Spot Intensity")
    axes[idx].set_xlabel("Mean Spot Intensity")
    if idx == 0:
        axes[idx].set_ylabel("PhTau Mean Intensity (per cell)")
    else:
        axes[idx].set_ylabel("")

plt.tight_layout()
plt.show()

# ==============================================================================
# 5. PRINT CORRELATION MATRIX PER DATASET
# ==============================================================================
for name, df_target in zip(["Dataset 1", "Dataset 2"], [df1_ints, df2_ints]):
    print(
        f"\n============= {name.upper()} CORRELATION MATRIX (INTENSITIES) ============="
    )
    analysis_cols = target_metrics + phtau_metrics
    corr_matrix = df_target[analysis_cols].corr(method="pearson")
    print(corr_matrix.loc[target_metrics, phtau_metrics].round(3))

## Analysis by annotation

### Import annotations

In [ ]:
import csv
import numpy as np
from skimage import io
from scipy.ndimage import binary_fill_holes

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================
CLUSTER_CSV = "Lysosome_annotations_combined.csv"
IMAGE_PATH = "Lysosome_masked_crops_1024.tif"  # Or your corresponding tif file
MIN_CELL_AREA = 10000

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def compute_valid_mask_indices(image_stack, area_threshold):
    """Vectorised masking and area filter over a standardised image stack."""
    raw_masks = np.any(image_stack > 0, axis=-1) if image_stack.ndim == 4 else image_stack > 0
    cell_masks = binary_fill_holes(raw_masks)
    cell_areas = np.sum(cell_masks, axis=(1, 2))
    return cell_areas >= area_threshold

# ==========================================
# 3. PIPELINE SYNCHRONIZATION
# ==========================================
# 1. Load the original image stack to deduce which frame indices survived filtering
raw_images_stack = io.imread(IMAGE_PATH)

# Standardise 4D stacks to always be (frames, spatial_1, spatial_2, channels)
if raw_images_stack.ndim == 4:
    channel_axis = np.argmin(raw_images_stack.shape)
    if channel_axis != -1:
        raw_images_stack = np.moveaxis(raw_images_stack, channel_axis, -1)

# 2. Generate the exact boolean mask array used by your image data pipeline
# True means the cell survived, False means it was dropped due to size
survived_import_mask = compute_valid_mask_indices(raw_images_stack, MIN_CELL_AREA)

# ==========================================
# 4. PARSE & FILTER ANNOTATIONS
# ==========================================
annotations = []
valid_indices = []

with open(CLUSTER_CSV, mode="r", encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)
    
    for i, row in enumerate(reader):
        # A. FIRST CHECK: Did this cell index survive the image import drop step?
        # If the image index is out of bounds or marked False in the mask, skip it.
        if i >= len(survived_import_mask) or not survived_import_mask[i]:
            continue

        # B. SECOND CHECK: Filter out "NoClass" (annotation -1.0)
        if row['annotation_names'] == "NoClass" or float(row['annotations']) == -1.0:
            continue
            
        # C. Extract relevant data
        ann_name = row['annotation_names']
        phtau_val = row['PhTau_cluster']
        treatment_val = row['treatment']
        
        # D. Create descriptive label
        phtau_status = "PhTau+" if phtau_val == '1' else "PhTau-"
        full_label = f"{ann_name}, {phtau_status}, {treatment_val}"
        
        # Store label and map its strict index configuration
        annotations.append([row['annotations'], full_label, ann_name, treatment_val, phtau_status])
        valid_indices.append(i)

filtered_images = [raw_images_stack[idx] for idx in valid_indices]

full_coord = load_pickle_scale("processed_data/spot_coordinates.pkl", TARGET_SCALE)
filtered_coord_map = [full_coord[idx] for idx in valid_indices]

### Cell areas

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.ndimage import binary_fill_holes
import seaborn as sns

# Define a consistent colour dictionary for the six specific annotations
annotation_colours = {
    'big clusters': '#F1C40F',          # Yellow
    'dispersed': '#3498DB',             # Blue
    'enlarged': '#E74C3C',              # Red
    'enlarged&dispersed': '#9B59B6',    # Purple
    'normal': '#2ECC71',                # Green
    'other/atypical': '#95A5A6'         # Grey
}

annotation_order = [
    'normal',
    'enlarged',
    'dispersed',
    'enlarged&dispersed',
    'big clusters',
    'other/atypical',
]

# Pixel-to-micrometer conversion factor (µm² per pixel)
pixel_width = 0.0739859
area_conversion = pixel_width ** 2

# 1. Compute the pixel area for each surviving image crop in Dataset 1
areas = []
for img in filtered_images:
    mask = np.any(img > 0, axis=-1) if img.ndim == 3 else img > 0
    filled = binary_fill_holes(mask)
    areas.append(np.sum(filled))

# 2. Build the DataFrame and convert areas to µm²
df1 = pd.DataFrame({
    'Area (µm²)': [a * area_conversion for a in areas],
    'Condition': [label[2] for label in annotations]
}).dropna()

In [ ]:
# 3. Plot the violin plot for Dataset 1
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df1,
    x='Condition',
    y='Area (µm²)',
    hue='Condition',
    palette=annotation_colours,
    legend=False,
    order=annotation_order,
    cut=1
)

ax.set_title('Cell Areas', fontsize=14, pad=10)
ax.set_ylabel('Area (µm²)', fontsize=12)
ax.set_xlabel('Annotation', fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

# Tilt x-axis labels and align them properly
plt.xticks(rotation=30, ha='right')

plt.savefig("Figures/areas_by_annotation.svg", format="svg", bbox_inches="tight")
plt.show()

# 4. Print Summary Statistics (Rounded to 2 decimal places for µm² precision)
print("--- Dataset 1 Summary Statistics (µm²) ---")
print(df1.groupby('Condition')['Area (µm²)'].describe().round(2))

In [ ]:
# ==============================================================================
# 5. STATISTICAL TESTS FOR CELL AREA BY ANNOTATION
# ==============================================================================

annotation_area_results = run_group_tests(
    df=df1,
    value_col="Area (µm²)",
    dataset_name="Dataset 1",
    metric_name="Cell area by annotation",
    condition_order=annotation_order
)

if annotation_area_results is not None:
    print("\nPairwise annotation-area results:")
    print(annotation_area_results)

### Lysosome counts

In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.ndimage import binary_fill_holes

# ==============================================================================
# 1. HELPER FUNCTION TO COMPUTE LYSOSOME COUNTS FROM COORD MAP
# ==============================================================================
def compute_lysosome_counts_from_coords(images_list, coord_map, phenotype_labels, dataset_name):
    rows = []
    
    for cell_id, (img, img_spots, phenotype) in enumerate(zip(images_list, coord_map, phenotype_labels)):
        # Compute cell area from the image mask
        raw_mask = np.any(img > 0, axis=-1)
        cell_mask = binary_fill_holes(raw_mask)
        cell_area = np.sum(cell_mask)        
        
        # Measure lysosome count directly from the coordinate map entries
        lys_count = len(img_spots) if img_spots is not None else 0
        
        cell_metrics = {
            "cell_id": cell_id, 
            "Annotation": phenotype, 
            "cell_area_px": cell_area,
            "lysosome_count": lys_count
        }
        rows.append(cell_metrics)

    return pd.DataFrame(rows)

# ==============================================================================
# 2. PREPARE DATASET 1 WITH ANNOTATIONS ONLY
# ==============================================================================
phenotype_labels = [label[2] for label in annotations]

df_counts = compute_lysosome_counts_from_coords(
    filtered_images, filtered_coord_map, phenotype_labels, dataset_name="Dataset 1 (Annotations Only)"
)

annotation_order = ['normal', 'enlarged', 'dispersed', 'enlarged&dispersed', 'big clusters', 'other/atypical']

In [ ]:
# ==============================================================================
# 3. GENERATE VIOLIN PLOT FOR LYSOSOME COUNTS
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_counts,
    x="Annotation",
    y="lysosome_count",
    hue="Annotation",
    order=annotation_order,
    hue_order=annotation_order,
    palette=annotation_colours,
    legend=False,
    cut=1,
)

ax.set_title("Lysosome Counts", fontsize=14, pad=15)
ax.set_ylabel("Lysosomes per Cell", fontsize=12)
ax.set_xlabel("Annotation", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=30, ha='right')

plt.savefig("Figures/lysosome_counts_by_annotation.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 4. PRINT SUMMARY STATISTICS
# ==============================================================================
print("\n============= LYSOSOME COUNT SUMMARY STATISTICS =============")
print(df_counts.groupby("Annotation")["lysosome_count"].describe().round(2))

In [ ]:
# ==============================================================================
# 5. STATISTICAL TESTS FOR LYSOSOME COUNTS BY ANNOTATION
# ==============================================================================

annotation_count_results = run_group_tests(
    df=df_counts,
    value_col="lysosome_count",
    dataset_name="Dataset 1",
    metric_name="Lysosome count by annotation",
    condition_order=annotation_order,
    group_col="Annotation"
)

if annotation_count_results is not None:
    print("\nPairwise annotation-count results:")
    print(annotation_count_results)

In [ ]:
pixel_width = 0.0739859
cell_area_um2 = np.array(areas) * (pixel_width ** 2)

df_counts["lysosome_density"] = df_counts["lysosome_count"] / cell_area_um2

fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_counts,
    x="Annotation",
    y="lysosome_density",
    hue="Annotation",
    order=annotation_order,
    hue_order=annotation_order,
    palette=annotation_colours,
    legend=False,
    cut=1,
)

ax.set_title("Lysosome Density", fontsize=14, pad=15)
ax.set_ylabel("Lysosomes per µm²", fontsize=12)
ax.set_xlabel("Annotation", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=30, ha='right')

plt.savefig("Figures/lysosome_density_by_annotation.svg", format="svg", bbox_inches="tight")
plt.show()

print("\n============= LYSOSOME DENSITY SUMMARY STATISTICS =============")
print(df_counts.groupby("Annotation")["lysosome_density"].describe().round(4))

In [ ]:
# ==============================================================================
# STATISTICAL TESTS FOR LYSOSOME DENSITY BY ANNOTATION
# ==============================================================================

annotation_density_results = run_group_tests(
    df=df_counts,
    value_col="lysosome_density",
    dataset_name="Dataset 1",
    metric_name="Lysosome density by annotation",
    condition_order=annotation_order,
    group_col="Annotation"
)

if annotation_density_results is not None:
    print("\nPairwise annotation-density results:")
    print(annotation_density_results)

### Lysosome coverage areas

In [ ]:
import csv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from skimage.filters import threshold_li, threshold_mean, threshold_otsu, threshold_triangle
from scipy.ndimage import binary_fill_holes

# ==============================================================================
# 1. HELPER FUNCTION TO RECOMPUTE COVERAGE WITH GLOBAL THRESHOLDS
# ==============================================================================
def compute_global_coverage(images_list, phenotype_labels, dataset_name, lysosome_channel=0):
    pooled_lys_pixels = []
    cell_masks = []
    lys_images = []

    # --- STEP 1: EXTRACT VALID LYSOSOME PIXELS & MASKS (LAST AXIS) ---
    for img in images_list:
        raw_mask = np.any(img > 0, axis=-1)
        cell_mask = binary_fill_holes(raw_mask)
        cell_masks.append(cell_mask)
        
        lys_img = img[..., lysosome_channel].astype(float)
        lys_images.append(lys_img)
        
        lys_pixels = lys_img[cell_mask]
        if lys_pixels.size > 0:
            pooled_lys_pixels.append(lys_pixels)

    all_non_zero_pixels = np.concatenate(pooled_lys_pixels)

    # --- STEP 2: CALCULATE GLOBAL THRESHOLDS ---
    global_thresholds = {
        "li_coverage_pct": threshold_li(all_non_zero_pixels),
        "otsu_coverage_pct": threshold_otsu(all_non_zero_pixels),
        "triangle_coverage_pct": threshold_triangle(all_non_zero_pixels),
        "mean_coverage_pct": threshold_mean(all_non_zero_pixels),
    }

    print(f"\n==== GLOBAL THRESHOLDS FOR {dataset_name.upper()} ====")
    print(global_thresholds)
    print("==========================================")

    # --- STEP 3: COMPUTE METRICS PER CELL ---
    rows = []
    for cell_id, (cell_mask, lys_img, phenotype) in enumerate(zip(cell_masks, lys_images, phenotype_labels)):
        cell_area = np.sum(cell_mask)        
        cell_metrics = {"cell_id": cell_id, "Annotation": phenotype, "cell_area_px": cell_area}

        if cell_area == 0:
            for metric_col in global_thresholds.keys():
                cell_metrics[metric_col] = 0.0
            rows.append(cell_metrics)
            continue

        for metric_col, thresh_value in global_thresholds.items():
            lys_mask = (lys_img > thresh_value) & cell_mask
            coverage_pct = (np.sum(lys_mask) / cell_area) * 100
            cell_metrics[metric_col] = coverage_pct

        rows.append(cell_metrics)

    return pd.DataFrame(rows)

# ==============================================================================
# 2. PREPARE DATASET 1 WITH ANNOTATIONS ONLY
# ==============================================================================
phenotype_labels = [label[2] for label in annotations]

df1 = compute_global_coverage(
    filtered_images, phenotype_labels, dataset_name="Dataset 1 (Annotations Only)"
)

target_metrics = {
    "li_coverage_pct": "Li",
    "otsu_coverage_pct": "Otsu",
    "triangle_coverage_pct": "Triangle",
    "mean_coverage_pct": "Mean",
}

annotation_order = ['normal', 'enlarged', 'dispersed', 'enlarged&dispersed', 'big clusters', 'other/atypical']

In [ ]:
# ==============================================================================
# 3. GENERATE COMPREHENSIVE PLOT (SINGLE COLUMN)
# ==============================================================================
fig, axes = plt.subplots(4, 1, figsize=(5, 12), sharex=True)

for row_idx, (metric_col, method_name) in enumerate(target_metrics.items()):
    ax = axes[row_idx]
    sns.violinplot(
        ax=ax,
        data=df1,
        x="Annotation",
        y=metric_col,
        hue="Annotation",
        order=annotation_order,
        hue_order=annotation_order,
        palette=annotation_colours,
        legend=False,
        cut=0
    )
    ax.set_ylabel(f"{method_name}\nCoverage (%)", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

axes[0].set_title(
    "Lysosome Coverage", fontsize=14, pad=10
)

for ax in axes:
    ax.set_xlabel("")
axes[-1].set_xlabel("Annotation", fontsize=12)

plt.xticks(rotation=30, ha='right')

plt.savefig("Figures/lysosome_coverage_by_annotation.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 4. PRINT SUMMARY STATISTICS
# ==============================================================================
for metric_col, method_name in target_metrics.items():
    print(f"\n============= {method_name.upper()} COVERAGE METRICS =============")
    print(df1.groupby("Annotation")[metric_col].describe().round(2))

In [ ]:
# ==============================================================================
# 5. STATISTICAL TESTS FOR COVERAGE BY ANNOTATION
# ==============================================================================

annotation_coverage_results = []

for metric_col, method_name in target_metrics.items():

    print(f"\n\n{'#' * 80}")
    print(f"{method_name} coverage by annotation")
    print(f"{'#' * 80}")

    result = run_group_tests(
        df=df1,
        value_col=metric_col,
        dataset_name="Dataset 1",
        metric_name=f"{method_name} coverage (%) by annotation",
        condition_order=annotation_order,
        group_col="Annotation"
    )

    if result is not None:
        annotation_coverage_results.append(result)


# Combine all pairwise results into one DataFrame
if annotation_coverage_results:
    annotation_coverage_results = pd.concat(
        annotation_coverage_results,
        ignore_index=True
    )

    print("\n\nAll annotation-based coverage results:")
    print(annotation_coverage_results)
else:
    annotation_coverage_results = pd.DataFrame()
    print("No pairwise results were generated.")

In [ ]:
# ==============================================================================
# GENERATE LI COVERAGE BY ANNOTATION PLOT
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df1,
    x="Annotation",
    y="li_coverage_pct",
    hue="Annotation",
    order=annotation_order,
    hue_order=annotation_order,
    palette=annotation_colours,
    legend=False,
    cut=0
)

ax.set_title("Lysosome Coverage", fontsize=14, pad=15)
ax.set_ylabel("Li Coverage (%)", fontsize=12)
ax.set_xlabel("Annotation", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.xticks(rotation=30, ha='right')

plt.savefig("Figures/li_coverage_by_annotation.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# PRINT SUMMARY STATISTICS
# ==============================================================================
print("\n============= LI COVERAGE SUMMARY STATISTICS =============")
print(df1.groupby("Annotation")["li_coverage_pct"].describe().round(2))

### Lysosome size

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# ==============================================================================
# 0. CONVERSION SETUP
# ==============================================================================
pixel_width = 0.0739859
area_conversion = pixel_width**2

# ==============================================================================
# 1. PREPARE INPUTS AND RUN THE LYSOSOME SIZE PROCESSING FUNCTION
# ==============================================================================
# Extract lysosome counts directly from filtered_coord_map
lysosomes_list = [
    len(spots) if spots is not None else 0 for spots in filtered_coord_map
]

# Process dataset 1 using the provided function
df_avg_size_1 = process_dataset_with_lysosomes(
    images_list=filtered_images,
    labels_list=phenotype_labels,
    lysosomes_list=lysosomes_list,
    global_thresholds=shared_thresholds,
    lysosome_channel=0,
)

target_avg_size_metrics = {
    "li_avg_size": "Li",
}

# Convert target size metric from pixels to µm²
for metric_col in target_avg_size_metrics.keys():
  if metric_col in df_avg_size_1.columns:
    df_avg_size_1[metric_col] = df_avg_size_1[metric_col] * area_conversion

In [ ]:
# ==============================================================================
# 2. GENERATE VIOLIN PLOT FOR LI AVERAGE LYSOSOME SIZE
# ==============================================================================
fig, ax = plt.subplots(figsize=(5, 3))

for metric_col, method_name in target_avg_size_metrics.items():
    sns.violinplot(
        ax=ax,
        data=df_avg_size_1,
        x="Condition",
        y=metric_col,
        hue="Condition",
        order=annotation_order,
        hue_order=annotation_order,
        palette=annotation_colours,
        legend=False,
        cut=0.5,
    )
    ax.set_ylabel(f"{method_name} Avg Size (µm²)", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

ax.set_title(
    "Lysosome Size",
    fontsize=14,
    pad=10,
)

ax.set_xlabel("Annotation", fontsize=12)

plt.xticks(rotation=30, ha='right')

plt.savefig("Figures/li_avg_size_by_annotation.svg", format="svg", bbox_inches="tight")
plt.show()

# ==============================================================================
# 3. PRINT SUMMARY STATISTICS
# ==============================================================================
for metric_col, method_name in target_avg_size_metrics.items():
    print(
        f"\n============= {method_name.upper()} AVERAGE LYSOSOME SIZE SUMMARY"
        f" (µm²) ============="
    )
    print(df_avg_size_1.groupby("Condition")[metric_col].describe().round(2))

In [ ]:
# ==============================================================================
# 4. STATISTICAL TESTS FOR AVERAGE LYSOSOME SIZE BY CONDITION
# ==============================================================================

avg_size_test_results = []

for metric_col, method_name in target_avg_size_metrics.items():

    print(f"\n\n{'#' * 80}")
    print(f"{method_name} average lysosome size")
    print(f"{'#' * 80}")

    result = run_group_tests(
        df=df_avg_size_1,
        value_col=metric_col,
        dataset_name="Dataset 1",
        metric_name=f"{method_name} average lysosome size (µm²)",
        condition_order=annotation_order,
        group_col="Condition"
    )

    if result is not None:
        avg_size_test_results.append(result)


# Combine all pairwise results
if avg_size_test_results:
    avg_size_test_results = pd.concat(
        avg_size_test_results,
        ignore_index=True
    )

    print("\n\nAll average-lysosome-size test results:")
    print(avg_size_test_results)
else:
    avg_size_test_results = pd.DataFrame()
    print("No pairwise results were generated.")

### Lysosome spot intensity

In [ ]:
# ==============================================================================
# LYSOSOME SPOT INTENSITY BY ANNOTATION
# ==============================================================================

# Extract spot intensities only from the cells retained in filtered_coord_map
spot_intensities_1 = extract_spot_intensities(
    filtered_images,
    filtered_coord_map,
    coord_order="xy",
)

# Calculate one mean spot intensity per cell
intensities_1 = get_mean_spot_intensity(spot_intensities_1)

# Confirm that the filtered cells and annotation labels remain aligned
assert len(intensities_1) == len(phenotype_labels), (
    f"Mismatch: {len(intensities_1)} intensity values but "
    f"{len(phenotype_labels)} annotation labels"
)

# Build the dataframe using the same annotation labels as df_avg_size_1
df_spot_intensity_1 = pd.DataFrame(
    {
        "Mean_Spot_Intensity": intensities_1,
        "Condition": phenotype_labels,
    }
).dropna(
    subset=["Mean_Spot_Intensity", "Condition"]
)

# ==============================================================================
# FILTER ANNOTATION ORDER
# ==============================================================================

annotation_order_present = [
    annotation
    for annotation in annotation_order
    if annotation in df_spot_intensity_1["Condition"].unique()
]

annotation_colours_present = {
    annotation: annotation_colours[annotation]
    for annotation in annotation_order_present
}

# ==============================================================================
# GENERATE VIOLIN PLOT
# ==============================================================================

fig, ax = plt.subplots(figsize=(5, 3))

sns.violinplot(
    ax=ax,
    data=df_spot_intensity_1,
    x="Condition",
    y="Mean_Spot_Intensity",
    hue="Condition",
    order=annotation_order_present,
    hue_order=annotation_order_present,
    palette=annotation_colours_present,
    legend=False,
    cut=0,
)

ax.set_title(
    "Lysosome Intensity",
    fontsize=14,
    pad=15,
)

ax.set_xlabel("Annotation", fontsize=12)
ax.set_ylabel("Mean Intensity per Cell", fontsize=12)
ax.grid(axis="y", linestyle="--", alpha=0.5)

ax.ticklabel_format(
    style="sci",
    axis="y",
    scilimits=(0, 0),
)

plt.xticks(rotation=30, ha="right")

plt.savefig(
    "Figures/mean_spot_intensity_by_annotation.svg",
    format="svg",
    bbox_inches="tight",
)

plt.show()

# ==============================================================================
# SUMMARY STATISTICS
# ==============================================================================

print("\n============= SPOT INTENSITY SUMMARY STATISTICS =============")
print(
    df_spot_intensity_1
    .groupby("Condition")["Mean_Spot_Intensity"]
    .describe()
    .reindex(annotation_order_present)
    .round(1)
)

In [ ]:
# ==============================================================================
# STATISTICAL TESTS FOR MEAN SPOT INTENSITY BY ANNOTATION
# ==============================================================================

annotation_intensity_results = run_group_tests(
    df=df_spot_intensity_1,
    value_col="Mean_Spot_Intensity",
    dataset_name="Dataset 1",
    metric_name="Mean lysosome spot intensity by annotation",
    condition_order=annotation_order_present,
    group_col="Condition"
)

if annotation_intensity_results is not None:
    print("\nPairwise annotation-intensity results:")
    print(annotation_intensity_results)